# Behavior test and Others
- Refer to __ Pechuck et. al. 2022<br>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tif
import networkx as nx
import os
import copy
import pickle
import itertools
import scipy

from tqdm import tqdm
import scipy.stats as stats
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter
from scipy.stats import linregress
import matplotlib.colors as colors
from matplotlib import cm

# Load data

In [ ]:
# common type

com_fname = r'your_directory\230724_neurontype_new.csv'    # new common id (230724 -)
com_id_df = pd.read_csv(com_fname)
com_id = np.array(com_id_df[['Neuron','Common_id']])
    # important!!!
    # Cook vs. Witvliet
com_type = np.array(com_id_df['Cook_typecode'])    # change


In [ ]:
# get index (230724 ver.)
# ========================================================================================================================
com_L_idx = [0,2,4,6,8,10,12,14,16,18,20,22,23,25,27,28,30,32,34,36,38,40,42,44,46,48,50,52,54,56,58,59,60,62,64,66,68,70,\
             72,74,75,76,78,80,82,84,86,88,90,92,94,96,98,100,102,104,106,108,109,110,112,114,116,117,119,121,122,124,126,\
             127,128,130,132,134,136,137,139,140,142,144,146,148,150,152,154,156,158,160,162,164,166,168,170,172,174,176,\
             178,180,181,182,183,184,185,186,187,196,197,198,199,200,201,202,203,212,214,216,218,220]
com_R_idx = [1,3,5,7,9,11,13,15,17,19,21,22,24,26,27,29,31,33,35,37,39,41,43,45,47,49,51,53,55,57,58,59,61,63,65,67,69,71,\
             73,74,75,77,79,81,83,85,87,89,91,93,95,97,99,101,103,105,107,108,109,111,113,115,116,118,120,121,123,125,126,\
             127,129,131,133,135,136,138,139,141,143,145,147,149,151,153,155,157,159,161,163,165,167,169,171,173,175,177,\
             179,188,189,190,191,192,193,194,195,204,205,206,207,208,209,210,211,213,215,217,219,221]

com_D_idx = [22,27,58,59,70,71,74,75,80,81,86,87,94,95,108,109,116,121,126,127,130,131,136,146,147,152,153,\
             156,157,160,161,164,165,168,169,176,177,180,181,182,183,184,185,186,187,188,189,190,191,192,\
             193,194,195,212,213,216,217]
com_V_idx = [22,27,58,59,72,73,74,75,84,85,90,91,96,97,108,109,116,121,126,127,134,135,139,148,149,154,155,\
             158,159,162,163,166,167,170,171,178,179,196,197,198,199,200,201,202,203,204,205,206,207,208,\
             209,210,211,214,215,220,221]
# ========================================================================================================================
com_L_idx = np.array(com_L_idx); com_R_idx = np.array(com_R_idx)
com_D_idx = np.array(com_D_idx); com_V_idx = np.array(com_V_idx)

# L/R/D/V id
LR_id = ['ADA','ADE','ADF','ADL','AFD','AIA','AIB','AIM','AIN','AIY','AIZ','ALA','ALM','ALN','AQR','ASE','ASG',\
         'ASH','ASI','ASJ','ASK','AUA','AVA','AVB','AVD','AVE','AVF','AVH','AVJ','AVK','AVL','AVM','AWA','AWB',\
         'AWC','BAG','BDU','CEPD','CEPV','DVA','DVC','FLP','HSN','IL1D','IL1','IL1V','IL2D','IL2','IL2V','OLL',\
         'OLQD','OLQV','PLN','PVC','PVN','PVP','PVQ','PVR','PVT','RIA','RIB','RIC','RID','RIF','RIG','RIH',\
         'RIM','RIP','RIR','RIS','RIV','RMDD','RMD','RMDV','RMED','RME','RMEV','RMF','RMG','RMH','SAAD','SAAV',\
         'SDQ','SIAD','SIAV','SIBD','SIBV','SMBD','SMBV','SMDD','SMDV','URAD','URAV','URB','URX','URYD','URYV']

DV_id = ['ALA','AQR','AVL','AVM','CEP L','CEP R','DVA','DVC','IL1 L','IL1 R','IL2 L','IL2 R','OLQ L','OLQ R',\
         'PVR','PVT','RID','RIH','RIR','RIS','RMD L','RMD R','RME','SAA L','SAA R','SIA L','SIA R',\
         'SIB L','SIB R','SMB L','SMB R','SMD L','SMD R','URA L','URA R','URY L','URY R']


## Connectivity matrices

In [ ]:
# normalized matrix
    # from matrix by HSY (230724 compilation ver.)

norm180_comp = np.load(r"your_directory\norm180_11_compilation_230724.npy", allow_pickle=True); print(norm180_comp.shape)
    # fixed 230724
norm222_comp = np.load(r"your_directory\norm222_11_compilation_230724.npy", allow_pickle=True); print(norm222_comp.shape,"\n")
    # fixed 230724

# binarize
norm180_comp_bi = np.copy(norm180_comp); norm180_comp_bi[norm180_comp_bi!=0] = 1
norm222_comp_bi = np.copy(norm222_comp); norm222_comp_bi[norm222_comp_bi!=0] = 1


# others (original) - 230724 ver. (gap junctions debugged on 240413)
n2u_chem_mat = np.load(r"your_directory\n2u_chem_common_230724.npy", allow_pickle=True); print(n2u_chem_mat.shape)
male_chem_mat = np.load(r"your_directory\male_chem_common_230724.npy", allow_pickle=True); print(male_chem_mat.shape)
dauer_chem_mat = np.load(r"your_directory\dauer_chem_common_230724.npy", allow_pickle=True); print(dauer_chem_mat.shape)

n2u_gap_mat = np.load(r"your_directory\n2u_gap_ori_180_230724.npy", allow_pickle=True); print(n2u_gap_mat.shape)
male_gap_mat = np.load(r"your_directory\male_gap_common_230724.npy", allow_pickle=True); print(male_gap_mat.shape)
dauer_gap_mat = np.load(r"your_directory\dauer1_gap_ori_180_230724.npy", allow_pickle=True); print(dauer_gap_mat.shape)


# Main

### Figure 2A circuit replicate
> (from Methods) <br>
> Connectivity data was taken from Cook et al. The strength of connectivity between each pair of neurons was described as the
total number of electron-microscopy sections containing that connection. For the simulation, we used the average of the corresponding
synaptic connections of the neurons in the left side and right side of the worm (ASH, AVA, AVB, AVD, PVC neurons) from
Cook et al., and each neuronal group (A and B neurons), and they were treated as a single cell. Connections within a neuronal
class were ignored. Chemical synapses between neural groups or pairs were simulated if they appeared in more than 10 serial
sections.

In [ ]:
# extract submatrix and apply 'Methods' from Pechuck 2022

# submatrix
noci_idx = [32,33, 42,43, 44,45, 46,47, 100,101]  # 230724 ver.
    # target neurons; ASH, AVA, AVB, AVD, PVC (LR pair)

n2u_noci_chem = np.copy(n2u_chem_mat[np.ix_(noci_idx,noci_idx)])
n2u_noci_gap = np.copy(n2u_gap_mat[np.ix_(noci_idx,noci_idx)])
male_noci_chem = np.copy(male_chem_mat[np.ix_(noci_idx,noci_idx)])
male_noci_gap = np.copy(male_gap_mat[np.ix_(noci_idx,noci_idx)])
dauer_noci_chem = np.copy(dauer_chem_mat[np.ix_(noci_idx,noci_idx)])
dauer_noci_gap = np.copy(dauer_gap_mat[np.ix_(noci_idx,noci_idx)])

# LR pair merge
n2u_noci_chem_merge = np.zeros((5,5)); n2u_noci_gap_merge = np.zeros((5,5))  # change size
male_noci_chem_merge = np.zeros((5,5)); male_noci_gap_merge = np.zeros((5,5))  # change size
dauer_noci_chem_merge = np.zeros((5,5)); dauer_noci_gap_merge = np.zeros((5,5))  # change size


In [ ]:
# chemical
for ii in range(n2u_noci_chem_merge.shape[0]):
    for jj in range(n2u_noci_chem_merge.shape[1]):
        square = n2u_noci_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2(or 4?) vs. np.nonzero(square)[0].size
        if ii==jj:
            continue  # ignore self
        else:
            n2u_noci_chem_merge[ii,jj] = square_avg

for ii in range(male_noci_chem_merge.shape[0]):
    for jj in range(male_noci_chem_merge.shape[1]):
        square = male_noci_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            male_noci_chem_merge[ii,jj] = square_avg

for ii in range(dauer_noci_chem_merge.shape[0]):
    for jj in range(dauer_noci_chem_merge.shape[1]):
        square = dauer_noci_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_noci_chem_merge[ii,jj] = square_avg


# gap junction
for ii in range(n2u_noci_gap_merge.shape[0]):
    for jj in range(n2u_noci_gap_merge.shape[1]):
        square = n2u_noci_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue  # ignore self
        else:
            n2u_noci_gap_merge[ii,jj] = square_avg

for ii in range(male_noci_gap_merge.shape[0]):
    for jj in range(male_noci_gap_merge.shape[1]):
        square = male_noci_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            male_noci_gap_merge[ii,jj] = square_avg

for ii in range(dauer_noci_gap_merge.shape[0]):
    for jj in range(dauer_noci_gap_merge.shape[1]):
        square = dauer_noci_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_noci_gap_merge[ii,jj] = square_avg


In [ ]:
# check full
print_df = pd.DataFrame(data=dauer_noci_gap,\
                        index=[com_id[noci_idx,0]], columns=[com_id[noci_idx,0]])
print_df

In [ ]:
# check merged
print_df = pd.DataFrame(data=dauer_noci_gap_merge,\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

### Normalize according to 'norm180_comp' ver.

In [ ]:
# from normalized matrix
n2u_noci_chem_norm = np.copy(norm180_comp[:,:,9][np.ix_(noci_idx,noci_idx)])
dauer_noci_chem_norm = np.copy(norm180_comp[:,:,10][np.ix_(noci_idx,noci_idx)])

# LR pair merge
n2u_noci_chem_merge_norm = np.zeros((5,5))
dauer_noci_chem_merge_norm = np.zeros((5,5))

# chemical
for ii in range(n2u_noci_chem_merge_norm.shape[0]):
    for jj in range(n2u_noci_chem_merge_norm.shape[1]):
        square = n2u_noci_chem_norm[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
        if ii==jj:
            continue
        else:
            n2u_noci_chem_merge_norm[ii,jj] = square_avg

for ii in range(dauer_noci_chem_merge_norm.shape[0]):
    for jj in range(dauer_noci_chem_merge_norm.shape[1]):
        square = dauer_noci_chem_norm[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
        if ii==jj:
            continue
        else:
            dauer_noci_chem_merge_norm[ii,jj] = square_avg


In [ ]:
# reshape dauer nociception circuit into Pechuk order

new_order = [0,1,3,2,4]

# adjust strength
mult_norm = np.ones((5,5))  # factor to multiply to dauer normalized matrix

ori_mat_temp = np.copy(n2u_noci_chem_merge[np.ix_(new_order,new_order)])
norm_mat_temp = np.copy(n2u_noci_chem_merge_norm[np.ix_(new_order,new_order)])
for ii in range(mult_norm.shape[0]):
    for jj in range(mult_norm.shape[1]):
        if norm_mat_temp[ii,jj]==0:
            continue
        else:
            mult_norm[ii,jj] = ori_mat_temp[ii,jj] / norm_mat_temp[ii,jj]

print(mult_norm)  # find the most frequent value (except 1)


In [ ]:
# check dataframe (original order)
print("  N2U (Cook)")
print_df = pd.DataFrame(data=np.round(n2u_noci_chem_merge,3),\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

In [ ]:
# check dataframe (Pechuk order)
print("  N2U (Cook)")
print_df = pd.DataFrame(data=np.round(ori_mat_temp,3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

In [ ]:
# check dataframe (Pechuk order)
print("  N2U (norm.)")
print_df = pd.DataFrame(data=np.round(norm_mat_temp,3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

In [ ]:
# check dataframe (Pechuk order)
print("  N2U (scaled)")
print_df = pd.DataFrame(data=np.round(norm_mat_temp * mult_norm[0,1],3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

---

In [ ]:
com_id[[46,47],:]

In [ ]:
# temporary check

target_idx = [34,35, 46,47]  # ASI, AVD

n2u_target_chem = np.copy(n2u_chem_mat[np.ix_(target_idx,target_idx)])
n2u_target_gap = np.copy(n2u_gap_mat[np.ix_(target_idx,target_idx)])
dauer_target_chem = np.copy(dauer_chem_mat[np.ix_(target_idx,target_idx)])
dauer_target_gap = np.copy(dauer_gap_mat[np.ix_(target_idx,target_idx)])

# LR pair merge
n2u_target_chem_merge = np.zeros((2,2)); n2u_target_gap_merge = np.zeros((2,2))
dauer_target_chem_merge = np.zeros((2,2)); dauer_target_gap_merge = np.zeros((2,2))

# =================================================================================

# chemical
for ii in range(n2u_target_chem_merge.shape[0]):
    for jj in range(n2u_target_chem_merge.shape[1]):
        square = n2u_target_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2(or 4?) vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            n2u_target_chem_merge[ii,jj] = square_avg

for ii in range(dauer_target_chem_merge.shape[0]):
    for jj in range(dauer_target_chem_merge.shape[1]):
        square = dauer_target_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_target_chem_merge[ii,jj] = square_avg


# gap junction
for ii in range(n2u_target_gap_merge.shape[0]):
    for jj in range(n2u_target_gap_merge.shape[1]):
        square = n2u_target_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            n2u_target_gap_merge[ii,jj] = square_avg

for ii in range(dauer_target_gap_merge.shape[0]):
    for jj in range(dauer_target_gap_merge.shape[1]):
        square = dauer_target_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_target_gap_merge[ii,jj] = square_avg


In [ ]:
# check full
print_df = pd.DataFrame(data=n2u_target_gap,\
                        index=[com_id[target_idx,0]], columns=[com_id[target_idx,0]])
print_df

In [ ]:
# check merged
print_df = pd.DataFrame(data=n2u_target_gap_merge,\
                        index=["ASI","AVD"], columns=["ASI","AVD"])
print_df

---

In [ ]:
import extract_connectivity_df as ecd
import supporting_functions as sf

path_to_cook_file = r"your_directory\SI_5_cook_original.xlsx"

herm_chem_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "hermaphrodite chemical", header = None)
herm_gapj_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "hermaphrodite gap jn symmetric", header = None)
male_chem_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "male chemical", header = None)
male_gapj_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "male gap jn symmetric", header = None)

# =============================================================================
# Define neurons of interest:
# In circuit_neurons the keys are the name of the neural class, and the values are
# the regex expressions to find the cells in Cook's data.  
# =============================================================================
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]',\
                   'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

Sgap_herm_head,Schem_herm_head,Sgap_male_head,Schem_male_head = ecd.final_connectivity_matrix(herm_chem_full_connectivity,\
                                                                                              herm_gapj_full_connectivity,\
                                                                                              male_chem_full_connectivity,\
                                                                                              male_gapj_full_connectivity,\
                                                                                              circuit_neurons)


In [ ]:
# (check) - transposed
print_df = pd.DataFrame(data=np.round(Schem_herm_head.T,3),\
                        index=["ASH","AVA","AVD","AVB","PVC","A","B"],\
                        columns=["ASH","AVA","AVD","AVB","PVC","A","B"])
print_df

In [ ]:
# (original order)
print_df = pd.DataFrame(data=np.round(n2u_noci_chem_merge,3),\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

---

In [ ]:
# reshape dauer nociception circuit into Pechuk order
    # Original order = ASH, AVA, AVB, AVD, PVC
    #   Pechuk order = ASH, AVA, AVD, AVB, PVC
    # assume that connectivity to A and B is similar to hermaphrodite (i.e., use hermaphrodite's data)

new_order = [0,1,3,2,4]  # 2 and 3 switched

Schem_dauer_head = np.zeros((7,7))  # mind the direction (transpose)
Sgap_dauer_head = np.zeros((7,7))  # mind the direction (transpose)

# chemical
dauer_chem_adjusted = np.copy( (dauer_noci_chem_merge_norm[np.ix_(new_order,new_order)] * mult_norm[0,1]) )  # to Pechuk order
Schem_dauer_head[:5,:5] = np.copy(dauer_chem_adjusted.T)  # note the transpose
Schem_dauer_head[5:,:] = np.copy(Schem_herm_head[5:,:])  # A, B row (from herm.)
Schem_dauer_head[:,5:] = np.copy(Schem_herm_head[:,5:])  # A, B column (from herm.)

# electrical (don't need to adjust strength; strength by section number)
    # but need to apply 'new_order' as well
dauer_gap_adjusted = np.copy( dauer_noci_gap_merge[np.ix_(new_order,new_order)] )  # to Pechuk order
Sgap_dauer_head[:5,:5] = np.copy(dauer_gap_adjusted.T)
Sgap_dauer_head[5:,:] = np.copy(Sgap_herm_head[5:,:])
Sgap_dauer_head[:,5:] = np.copy(Sgap_herm_head[:,5:])


In [ ]:
# check (Pechuk order)
print("Dauer")
print_df_1 = pd.DataFrame(data=np.round(Sgap_dauer_head[:5,:5].T,2),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])  # Schem vs. Sgap
print_df_1

In [ ]:
# check (Pechuk order)
print("Herm.")
print_df_2 = pd.DataFrame(data=np.round(Sgap_herm_head[:5,:5].T,2),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])  # Schem vs. Sgap
print_df_2

In [ ]:
# to networkx graph

Schem_d_adj = np.copy(Schem_dauer_head[:5,:5].T)  # Note the transpose (due to networkx direction setting)
Schem_n_adj = np.copy(Schem_herm_head[:5,:5].T)  # Note the transpose (due to networkx direction setting)
Sgap_d_adj = np.copy(Sgap_dauer_head[:5,:5].T)
Sgap_n_adj = np.copy(Sgap_herm_head[:5,:5].T)


d_G_chem = nx.from_numpy_array(Schem_d_adj, parallel_edges=False, create_using=nx.DiGraph)  # directed
n_G_chem = nx.from_numpy_array(Schem_n_adj, parallel_edges=False, create_using=nx.DiGraph)

d_G_gap = nx.from_numpy_array(Sgap_d_adj, parallel_edges=False, create_using=nx.Graph)  # undirected
n_G_gap = nx.from_numpy_array(Sgap_n_adj, parallel_edges=False, create_using=nx.Graph)

noci_label_pechuk = ["ASH","AVA","AVD","AVB","PVC"]  # change
node_colors_pechuk = ['#FCB0E5','#D65E00','#D65E00','#D65E00','#D65E00']  # change

labels_pechuk = {}
for node in d_G_chem:
    labels_pechuk[node] = noci_label_pechuk[node]


### Dimorphism (Chemical, Gap)

In [ ]:
# row=pre; col=post
d_chem_temp = np.copy(Schem_dauer_head[:5,:5].T)  # Note the transpose
n_chem_temp = np.copy(Schem_herm_head[:5,:5].T)  # Note the transpose
d_gap_temp = np.copy(Sgap_dauer_head[:5,:5].T)
n_gap_temp = np.copy(Sgap_herm_head[:5,:5].T)

In [ ]:
# binarize

d_chem_temp_bi = np.copy(d_chem_temp); d_chem_temp_bi[d_chem_temp_bi!=0] = 1
n_chem_temp_bi = np.copy(n_chem_temp); n_chem_temp_bi[n_chem_temp_bi!=0] = 1
d_gap_temp_bi = np.copy(d_gap_temp); d_gap_temp_bi[d_gap_temp_bi!=0] = 1
n_gap_temp_bi = np.copy(n_gap_temp); n_gap_temp_bi[n_gap_temp_bi!=0] = 1


In [ ]:
# Dimorphism 1 - 'Binary' difference

# chemical
dimorph_bi_chem = d_chem_temp_bi - n_chem_temp_bi
di_bc_idx = np.where(dimorph_bi_chem!=0); print("Chemical:", di_bc_idx[0].size)

# gap
dimorph_bi_gap = d_gap_temp_bi - n_gap_temp_bi
di_bg_idx = np.where(np.triu(dimorph_bi_gap)!=0); print("Gap:", di_bg_idx[0].size)  # note the 'np.triu' function


In [ ]:
# print

N_name = ["ASH","AVA","AVD","AVB","PVC"]  # Pechuk order

print("Chemical")
for ii in range(di_bc_idx[0].size):
    I1=di_bc_idx[0][ii]; I2=di_bc_idx[1][ii]
    print(N_name[I1], " -> ", N_name[I2], " ", dimorph_bi_chem[I1,I2])

print("\nGap")
for ii in range(di_bg_idx[0].size):
    I1=di_bg_idx[0][ii]; I2=di_bg_idx[1][ii]
    print(N_name[I1], " == ", N_name[I2], " ", dimorph_bi_gap[I1,I2])


In [ ]:
# Dimorphism 2 - 'Fold' difference

f_thresh = 1.5

# chemical
dimorph_fold_chem = np.zeros((5,5))
for ii in range(dimorph_fold_chem.shape[0]):
    for jj in range(dimorph_fold_chem.shape[1]):
        val_d = d_chem_temp[ii,jj]; val_n = n_chem_temp[ii,jj]
        if (val_d!=0) and (val_n!=0):
            dimorph_fold_chem[ii,jj] = np.max([val_d,val_n]) / np.min([val_d,val_n])
        else:
            continue

di_fc_idx = np.where(dimorph_fold_chem > f_thresh); print("Chemical:", di_fc_idx[0].size)

# gap
dimorph_fold_gap = np.zeros((5,5))
for ii in range(dimorph_fold_gap.shape[0]):
    for jj in range(dimorph_fold_gap.shape[1]):
        val_d = d_gap_temp[ii,jj]; val_n = n_gap_temp[ii,jj]
        if val_d!=0 and val_n!=0:
            dimorph_fold_gap[ii,jj] = np.max([val_d,val_n]) / np.min([val_d,val_n])
        else:
            continue

di_fg_idx = np.where(np.triu(dimorph_fold_gap) > f_thresh); print("Gap:", di_fg_idx[0].size)  # note the 'np.triu' function


In [ ]:
# print

N_name = ["ASH","AVA","AVD","AVB","PVC"]  # Pechuk order

print("Chemical")
for ii in range(di_fc_idx[0].size):
    I1=di_fc_idx[0][ii]; I2=di_fc_idx[1][ii]
    print(N_name[I1], " -> ", N_name[I2], " ", dimorph_fold_chem[I1,I2])

print("\nGap")
for ii in range(di_fg_idx[0].size):
    I1=di_fg_idx[0][ii]; I2=di_fg_idx[1][ii]
    print(N_name[I1], " == ", N_name[I2], " ", dimorph_fold_gap[I1,I2])


---

In [ ]:
# (Adult Chemical, Pechuk order)
print_df = pd.DataFrame(data=np.round(n_chem_temp,3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

In [ ]:
# (Dauer Chemical, Pechuk order)
print_df = pd.DataFrame(data=np.round(d_chem_temp,3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

In [ ]:
# (Adult Gap, Pechuk order)
print_df = pd.DataFrame(data=np.round(n_gap_temp,3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

In [ ]:
# (Dauer Gap, Pechuk order)
print_df = pd.DataFrame(data=np.round(d_gap_temp,3),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])
print_df

### Characteristics of subnetworks

In [ ]:
# check subnetwork
    # 'Schem_n_adj' was already transposed (row: pre; column: post)

print("Herm.")
print_df_s = pd.DataFrame(data=np.round(Schem_n_adj,2),\
                          index=["ASH","AVA","AVD","AVB","PVC"],\
                          columns=["ASH","AVA","AVD","AVB","PVC"])
print_df_s


In [ ]:
# check subnetwork (row: pre; column: post)

print("Dauer")
print_df_s = pd.DataFrame(data=np.round(Schem_d_adj,2),\
                          index=["ASH","AVA","AVD","AVB","PVC"],\
                          columns=["ASH","AVA","AVD","AVB","PVC"])
print_df_s


In [ ]:
# weight sum

Schem_d_wsum = np.sum(Schem_d_adj)
Schem_n_wsum = np.sum(Schem_n_adj)
Sgap_d_wsum = np.sum(np.triu(Sgap_d_adj))
Sgap_n_wsum = np.sum(np.triu(Sgap_n_adj))


In [ ]:
# (plot)


b_width = 0.6; fsz=14; lsz=14


fig, ((ax1,ax2)) = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(5,4))
xpos = [1,2]

ax1.set_ylabel('Total connection weight', fontsize=14, fontname='sans-serif')
ax1.set_title('Chemical', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], Schem_n_wsum, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[1], Schem_d_wsum, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,162)
ax1.set_xticks(xpos)
ax1.set_xticklabels(["Adult","Dauer"], fontsize=fsz, rotation=0, fontname='sans-serif')
ax1.set_yticks([0,30,60,90,120,150])
ax1.set_yticklabels(["0","30","60","90","120","150"], fontsize=11.5, rotation=0, fontname='Arial')

ax2.set_title('Electrical', fontsize=18, fontname='sans-serif')
ax2.bar(xpos[0], Sgap_n_wsum, width=b_width, color="#1F77B4", alpha=0.7)
ax2.bar(xpos[1], Sgap_d_wsum, width=b_width, color="#FF7F0E", alpha=0.7)
ax2.set_xticks(xpos)
ax2.set_xticklabels(["Adult","Dauer"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.text(xpos[0], Schem_n_wsum, "{}".format(round(Schem_n_wsum,1)),\
         ha='center', va='bottom', color='k', fontsize=lsz, fontname='sans-serif')
ax1.text(xpos[1], Schem_d_wsum, "{}".format(round(Schem_d_wsum,1)),\
         ha='center', va='bottom', color='k', fontsize=lsz, fontname='sans-serif')
ax2.text(xpos[0], Sgap_n_wsum, "{}".format(round(Sgap_n_wsum,1)),\
         ha='center', va='bottom', color='k', fontsize=lsz, fontname='sans-serif')
ax2.text(xpos[1], Sgap_d_wsum, "{}".format(round(Sgap_d_wsum,1)),\
         ha='center', va='bottom', color='k', fontsize=lsz, fontname='sans-serif')

plt.show()


In [ ]:
# number of connections

Schem_d_cnum = np.nonzero(Schem_d_adj)[0].size
Schem_n_cnum = np.nonzero(Schem_n_adj)[0].size
Sgap_d_cnum = np.nonzero(np.triu(Sgap_d_adj))[0].size  # upper triangle only
Sgap_n_cnum = np.nonzero(np.triu(Sgap_n_adj))[0].size  # upper triangle only


In [ ]:
# (plot)


b_width = 0.6; fsz=14; lsz=14


fig, ((ax1,ax2)) = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(5,4))
xpos = [1,2]

ax1.set_ylabel('Total connection number', fontsize=14, fontname='sans-serif')
ax1.set_title('Chemical', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], Schem_n_cnum, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[1], Schem_d_cnum, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,13)
ax1.set_xticks(xpos)
ax1.set_xticklabels(["Adult","Dauer"], fontsize=fsz, rotation=0, fontname='sans-serif')
ax1.set_yticks([0,3,6,9,12])
ax1.set_yticklabels(["0","3","6","9","12"], fontsize=11.5, rotation=0, fontname='sans-serif')

ax2.set_title('Electrical', fontsize=18, fontname='Arial')
ax2.bar(xpos[0], Sgap_n_cnum, width=b_width, color="#1F77B4", alpha=0.7)
ax2.bar(xpos[1], Sgap_d_cnum, width=b_width, color="#FF7F0E", alpha=0.7)
ax2.set_xticks(xpos)
ax2.set_xticklabels(["Adult","Dauer"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.text(xpos[0], Schem_n_cnum, "{}".format(Schem_n_cnum),\
         ha='center', va='bottom', color='k', fontsize=lsz, fontname='sans-serif')
ax1.text(xpos[1], Schem_d_cnum, "{}".format(Schem_d_cnum),\
         ha='center', va='bottom', color='k', fontsize=lsz, fontname='sans-serif')
ax2.text(xpos[0], Sgap_n_cnum, "{}".format(Sgap_n_cnum),\
         ha='center', va='bottom', color='k', fontsize=lsz, fontname='sans-serif')
ax2.text(xpos[1], Sgap_d_cnum, "{}".format(Sgap_d_cnum),\
         ha='center', va='bottom', color='k', fontsize=lsz, fontname='sans-serif')

plt.show()


In [ ]:
# chemical
for ii in range(n2u_chem_merge.shape[0]):
    for jj in range(n2u_chem_merge.shape[1]):
        PRE_ID = [com_L_idx[ii], com_R_idx[ii]]
        POST_ID = [com_L_idx[jj], com_R_idx[jj]]
        
        square = n2u_chem_mat[np.ix_(PRE_ID, POST_ID)]
        square_avg = np.sum(square) / square.size
            # options for denominator = 2(or 4?) vs. np.nonzero(square)[0].size
        if ii==jj:
            continue  # ignore self
        else:
            n2u_chem_merge[ii,jj] = square_avg

for ii in range(dauer_chem_merge.shape[0]):
    for jj in range(dauer_chem_merge.shape[1]):
        PRE_ID = [com_L_idx[ii], com_R_idx[ii]]
        POST_ID = [com_L_idx[jj], com_R_idx[jj]]
        
        square = dauer_chem_norm[np.ix_(PRE_ID, POST_ID)]
        square_avg = np.sum(square) / square.size
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_chem_merge[ii,jj] = square_avg


# gap junction
for ii in range(n2u_gap_merge.shape[0]):
    for jj in range(n2u_gap_merge.shape[1]):
        PRE_ID = [com_L_idx[ii], com_R_idx[ii]]
        POST_ID = [com_L_idx[jj], com_R_idx[jj]]
        
        square = n2u_gap_mat[np.ix_(PRE_ID, POST_ID)]
        square_avg = np.sum(square) / square.size
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue  # ignore self
        else:
            n2u_gap_merge[ii,jj] = square_avg

for ii in range(dauer_gap_merge.shape[0]):
    for jj in range(dauer_gap_merge.shape[1]):
        PRE_ID = [com_L_idx[ii], com_R_idx[ii]]
        POST_ID = [com_L_idx[jj], com_R_idx[jj]]
        
        square = dauer_gap_mat[np.ix_(PRE_ID, POST_ID)]
        square_avg = np.sum(square) / square.size
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_gap_merge[ii,jj] = square_avg


## Contact check

In [ ]:
fname_contact = r"your_directory\total_contact_arranged_new.xlsx"

dauer_contact_df = pd.read_excel(fname_contact, sheet_name = "dauer_221021_norm", header = None)
dset1_contact_df = pd.read_excel(fname_contact, sheet_name = "dataset1_norm", header = None)
dset2_contact_df = pd.read_excel(fname_contact, sheet_name = "dataset2_norm", header = None)
dset3_contact_df = pd.read_excel(fname_contact, sheet_name = "dataset3_norm", header = None)
dset4_contact_df = pd.read_excel(fname_contact, sheet_name = "dataset4_norm", header = None)
dset5_contact_df = pd.read_excel(fname_contact, sheet_name = "dataset5_norm", header = None)
dset6_contact_df = pd.read_excel(fname_contact, sheet_name = "dataset6_norm", header = None)
dset8_contact_df = pd.read_excel(fname_contact, sheet_name = "dataset8_norm", header = None)

dauer_contact = np.copy(np.array(dauer_contact_df)[2:,2:]); print(dauer_contact.shape)
dset1_contact = np.copy(np.array(dset1_contact_df)[2:,2:]); print(dset1_contact.shape)
dset2_contact = np.copy(np.array(dset2_contact_df)[2:,2:]); print(dset2_contact.shape)
dset3_contact = np.copy(np.array(dset3_contact_df)[2:,2:]); print(dset3_contact.shape)
dset4_contact = np.copy(np.array(dset4_contact_df)[2:,2:]); print(dset4_contact.shape)
dset5_contact = np.copy(np.array(dset5_contact_df)[2:,2:]); print(dset5_contact.shape)
dset6_contact = np.copy(np.array(dset6_contact_df)[2:,2:]); print(dset6_contact.shape)
dset8_contact = np.copy(np.array(dset8_contact_df)[2:,2:]); print(dset8_contact.shape)

In [ ]:
wit_contact_full = np.zeros((222,222,7))
wit_contact_full[:,:,0] = np.copy(dset1_contact)
wit_contact_full[:,:,1] = np.copy(dset2_contact)
wit_contact_full[:,:,2] = np.copy(dset3_contact)
wit_contact_full[:,:,3] = np.copy(dset4_contact)
wit_contact_full[:,:,4] = np.copy(dset5_contact)
wit_contact_full[:,:,5] = np.copy(dset6_contact)
wit_contact_full[:,:,6] = np.copy(dset8_contact)

---

In [ ]:
com_id[[34,35],:]

In [ ]:
# neuron selection

contact_idx = [34,35, 46,47]  # ASI, AVD

dset8_target_contact = np.copy(dset8_contact[np.ix_(contact_idx,contact_idx)])
dauer_target_contact = np.copy(dauer_contact[np.ix_(contact_idx,contact_idx)])

# LR pair merge
dset8_target_contact_merge = np.zeros((2,2))
dauer_target_contact_merge = np.zeros((2,2))

# =================================================================================

for ii in range(dset8_target_contact_merge.shape[0]):
    for jj in range(dset8_target_contact_merge.shape[1]):
        square = dset8_target_contact[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2(or 4?) vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dset8_target_contact_merge[ii,jj] = square_avg

for ii in range(dauer_target_contact_merge.shape[0]):
    for jj in range(dauer_target_contact_merge.shape[1]):
        square = dauer_target_contact[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_target_contact_merge[ii,jj] = square_avg


In [ ]:
# check full
print_df = pd.DataFrame(data=dset8_target_contact,\
                        index=[com_id[contact_idx,0]], columns=[com_id[contact_idx,0]])
print_df

In [ ]:
# check full
print_df = pd.DataFrame(data=dauer_target_contact,\
                        index=[com_id[contact_idx,0]], columns=[com_id[contact_idx,0]])
print_df

In [ ]:
# check merged
print_df = pd.DataFrame(data=dset8_target_contact_merge,\
                        index=["ASI","AVD"], columns=["ASI","AVD"])
print_df

In [ ]:
# check merged
print_df = pd.DataFrame(data=dauer_target_contact_merge,\
                        index=["ASI","AVD"], columns=["ASI","AVD"])
print_df

# Behavior test - Reversal duration

In [ ]:
# read in excel file

noci_file = r'your_directory\nociception_test.xlsx'
noci_df = pd.read_excel(noci_file, sheet_name='Preliminary', header=None)
noci_np = np.array(noci_df)

n2_result = np.copy(noci_np[3:,0:4])
jn2389_result = np.copy(noci_np[3:,4:8])
jn1713_result = np.copy(noci_np[3:,8:12])

n2_pd_result = np.copy(noci_np[3:,14:16])


In [ ]:
# refine data
    # column 1 = reversal duration ('wave')
    # column 2 = reversed or not ('back'; 1 = reversal, 0 = no reversal)

n2_dauer = np.copy(n2_result[:41,:2]); n2_adult = np.copy(n2_result[:41,2:])
jn2389_dauer = np.copy(jn2389_result[:41,:2]); jn2389_adult = np.copy(jn2389_result[:41,2:])
jn1713_dauer = np.copy(jn1713_result[:41,:2]); jn1713_adult = np.copy(jn1713_result[:41,2:])

n2_pd_adult = np.copy(n2_pd_result[:,:])


In [ ]:
# reversal percentage

print("N2 :")
n2_dauer_rev_percent = (np.sum(n2_dauer[:,1]) / n2_dauer.shape[0]) * 100
n2_adult_rev_percent = (np.sum(n2_adult[:,1]) / n2_adult.shape[0]) * 100
print("Dauer :", n2_dauer_rev_percent); print("Adult :", n2_adult_rev_percent,"\n")

print("JN2389 :")
jn2389_dauer_rev_percent = (np.sum(jn2389_dauer[:,1]) / jn2389_dauer.shape[0]) * 100
jn2389_adult_rev_percent = (np.sum(jn2389_adult[:,1]) / jn2389_adult.shape[0]) * 100
print("Dauer :", jn2389_dauer_rev_percent); print("Adult :", jn2389_adult_rev_percent,"\n")

print("JN1713 :")
jn1713_dauer_rev_percent = (np.sum(jn1713_dauer[:,1]) / jn1713_dauer.shape[0]) * 100
jn1713_adult_rev_percent = (np.sum(jn1713_adult[:,1]) / jn1713_adult.shape[0]) * 100
print("Dauer :", jn1713_dauer_rev_percent); print("Adult :", jn1713_adult_rev_percent)


In [ ]:
# is_reversed index

n2_dauer_rev_idx = np.where(n2_dauer[:,1]==1)[0]
n2_adult_rev_idx = np.where(n2_adult[:,1]==1)[0]

jn2389_dauer_rev_idx = np.where(jn2389_dauer[:,1]==1)[0]
jn2389_adult_rev_idx = np.where(jn2389_adult[:,1]==1)[0]

jn1713_dauer_rev_idx = np.where(jn1713_dauer[:,1]==1)[0]
jn1713_adult_rev_idx = np.where(jn1713_adult[:,1]==1)[0]

n2_pd_adult_rev_idx = np.where(n2_pd_adult[:,1]==1)[0]


In [ ]:
# statistical test (only with reversed data)

print("Dauer (R)")
print(stats.ranksums(n2_dauer[n2_dauer_rev_idx,0],jn2389_dauer[jn2389_dauer_rev_idx,0])[1])
print(stats.ranksums(n2_dauer[n2_dauer_rev_idx,0],jn1713_dauer[jn1713_dauer_rev_idx,0])[1])
print(stats.ranksums(jn2389_dauer[jn2389_dauer_rev_idx,0],jn1713_dauer[jn1713_dauer_rev_idx,0])[1],"\n")

print("Adult (R)")
print(stats.ranksums(n2_adult[n2_adult_rev_idx,0],jn2389_adult[jn2389_adult_rev_idx,0])[1])
print(stats.ranksums(n2_adult[n2_adult_rev_idx,0],jn1713_adult[jn1713_adult_rev_idx,0])[1])
print(stats.ranksums(jn2389_adult[jn2389_adult_rev_idx,0],jn1713_adult[jn1713_adult_rev_idx,0])[1],"\n")

print("Dauer vs. Adult (R)")
print(stats.ranksums(n2_dauer[n2_dauer_rev_idx,0],n2_adult[n2_adult_rev_idx,0])[1])
print(stats.ranksums(jn2389_dauer[jn2389_dauer_rev_idx,0],jn2389_adult[jn2389_adult_rev_idx,0])[1])
print(stats.ranksums(jn1713_dauer[jn1713_dauer_rev_idx,0],jn1713_adult[jn1713_adult_rev_idx,0])[1])


In [ ]:
# plot


b_width = 0.5; v_width = 0.6; ssz=14; fsz=14


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel("number of sine waves", fontsize=18, fontname='sans-serif')
xpos = [1,2]


temp3 = np.copy(jn2389_adult[jn2389_adult_rev_idx,0])
temp4 = np.copy(jn2389_dauer[jn2389_dauer_rev_idx,0])


ax1.boxplot(temp3, positions=[xpos[0]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#1F77B4'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})
ax1.boxplot(temp4, positions=[xpos[1]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#FF7F0E'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})

ax1.set_xlim(0.4,2.6)
ax1.set_ylim(-0.1,4.2)
ax1.set_xticks(xpos)
ax1.set_xticklabels(["Adult","Dauer"], fontsize=14, rotation=0, fontname='sans-serif', ha='center')
ax1.set_yticks([0,1,2,3,4])
ax1.set_yticklabels(["0","1","2","3","4"], fontsize=14, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [3.7,3.7], lw=0.8, c='k')
ax1.text(1.5, 3.8, "***", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
fig.tight_layout(pad=0.4)

plt.show()


In [ ]:
# statistical test (only with reversed data)

temp0 = np.copy(n2_dauer[n2_dauer_rev_idx,0]).astype(np.float64)
temp1 = np.copy(n2_adult[n2_adult_rev_idx,0]).astype(np.float64)
temp2 = np.copy(n2_pd_adult[n2_pd_adult_rev_idx,0]).astype(np.float64)


print("Dauer vs. Adult (Rev.)")
print(stats.mannwhitneyu(temp0,temp1, method='exact')[1], "\n")

print("Dauer vs. PD-Adult (Rev.)")
print(stats.mannwhitneyu(temp0,temp2, method='exact')[1], "\n")

print("Adult vs. PD-Adult (Rev.)")
print(stats.mannwhitneyu(temp1,temp2, method='exact')[1])


In [ ]:
# mean ; error

# from scipy.stats import mannwhitneyu
# from scipy.stats import ranksums

temp1 = np.copy(n2_dauer[n2_dauer_rev_idx,0])
temp2 = np.copy(n2_adult[n2_adult_rev_idx,0])
temp3 = np.copy(n2_pd_adult[n2_pd_adult_rev_idx,0])


DA_RD_mean = np.mean(temp1)  # DAuer
AD_RD_mean = np.mean(temp2)  # ADult
PA_RD_mean = np.mean(temp3)  # Post-dauer Adult

DA_RD_sem = stats.sem(temp1)
AD_RD_sem = stats.sem(temp2)
PA_RD_sem = stats.sem(temp3)


In [ ]:
# bar graph


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3.2,4))

xpos = [1,2,3]

ax1.set_ylabel('number of sine waves', fontsize=14, fontname='sans-serif')
ax1.bar(xpos[0], DA_RD_mean, yerr=DA_RD_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.bar(xpos[1], AD_RD_mean, yerr=AD_RD_sem, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[2], PA_RD_mean, yerr=PA_RD_sem, width=b_width, color="#9467BD", alpha=0.7)
ax1.set_xlim(0.4,3.6)
ax1.set_ylim(0,2.8)
label = ["\nDauer","\nAdult","Post-dauer\nAdult"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=12.5, rotation=0, fontname='sans-serif')
ax1.set_yticks(np.arange(0,3.0,0.5))
ax1.set_yticklabels(["0","0.5","1.0","1.5","2.0","2.5"], fontsize=12.5, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [2.20,2.20], lw=0.8, c='k')
ax1.text(1.5, 2.21, "***", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
ax1.plot([0.9,3.1], [2.53,2.53], lw=0.8, c='k')
ax1.text(2.0, 2.54, "***", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')

plt.show()


## Different stimulus range for dauer and adult
- Same concentration to both dauer and adult may cause bias since dauer's mouth is closed. <br>
- And also to check whether concentration difference change reversal duration in dauer and adult.

In [ ]:
# read file

noci_file = r'your_directory\nociception_test.xlsx'

N2_diff_df = pd.read_excel(noci_file, sheet_name='N2_diff_stim', header=None)
N2_diff_np = np.array(N2_diff_df)
N2_diff_dauer_result = np.copy(N2_diff_np[4:,:10])  # 5 different concentration
N2_diff_adult_result = np.copy(N2_diff_np[4:,10:])  # 5 different concentration

JN2389_diff_df = pd.read_excel(noci_file, sheet_name='JN2389_diff_stim', header=None)
JN2389_diff_np = np.array(JN2389_diff_df)
JN2389_diff_dauer_result = np.copy(JN2389_diff_np[4:,:10])
JN2389_diff_adult_result = np.copy(JN2389_diff_np[4:,10:])

JN1713_diff_df = pd.read_excel(noci_file, sheet_name='JN1713_diff_stim', header=None)
JN1713_diff_np = np.array(JN1713_diff_df)
JN1713_diff_dauer_result = np.copy(JN1713_diff_np[4:,:10])
JN1713_diff_adult_result = np.copy(JN1713_diff_np[4:,10:])


In [ ]:
# reversal percentage

# N2
denom = np.sum(~pd.isnull(N2_diff_dauer_result[:,[1,3,5,7,9]]), axis=0)
n2_diff_dauer_rev_percent = ( np.nansum(N2_diff_dauer_result[:,[1,3,5,7,9]], axis=0) / denom ) * 100
denom = np.sum(~pd.isnull(N2_diff_adult_result[:,[1,3,5,7,9]]), axis=0)
n2_diff_adult_rev_percent = ( np.nansum(N2_diff_adult_result[:,[1,3,5,7,9]], axis=0) / denom ) * 100

# # JN2389
# denom = np.sum(~pd.isnull(JN2389_diff_dauer_result[:,[1,3,5,7,9]]), axis=0)
# jn2389_diff_dauer_rev_percent = ( np.nansum(JN2389_diff_dauer_result[:,[1,3,5,7,9]], axis=0) / denom ) * 100
# denom = np.sum(~pd.isnull(JN2389_diff_adult_result[:,[1,3,5,7,9]]), axis=0)
# jn2389_diff_adult_rev_percent = ( np.nansum(JN2389_diff_adult_result[:,[1,3,5,7,9]], axis=0) / denom ) * 100

# JN1713
denom = np.sum(~pd.isnull(JN1713_diff_dauer_result[:,[1,3,5,7,9]]), axis=0)
jn1713_diff_dauer_rev_percent = ( np.nansum(JN1713_diff_dauer_result[:,[1,3,5,7,9]], axis=0) / denom ) * 100
denom = np.sum(~pd.isnull(JN1713_diff_adult_result[:,[1,3,5,7,9]]), axis=0)
jn1713_diff_adult_rev_percent = ( np.nansum(JN1713_diff_adult_result[:,[1,3,5,7,9]], axis=0) / denom ) * 100


In [ ]:
# reversal duration - NaN cutoff

# N2
n2_diff_dauer_1 = N2_diff_dauer_result[:,0:2][~pd.isnull(N2_diff_dauer_result[:,0]),:]
n2_diff_dauer_2 = N2_diff_dauer_result[:,2:4][~pd.isnull(N2_diff_dauer_result[:,2]),:]
n2_diff_dauer_4 = N2_diff_dauer_result[:,4:6][~pd.isnull(N2_diff_dauer_result[:,4]),:]
n2_diff_dauer_6 = N2_diff_dauer_result[:,6:8][~pd.isnull(N2_diff_dauer_result[:,6]),:]
n2_diff_dauer_8 = N2_diff_dauer_result[:,8:][~pd.isnull(N2_diff_dauer_result[:,8]),:]
n2_diff_adult_002 = N2_diff_adult_result[:,0:2][~pd.isnull(N2_diff_adult_result[:,0]),:]
n2_diff_adult_004 = N2_diff_adult_result[:,2:4][~pd.isnull(N2_diff_adult_result[:,2]),:]
n2_diff_adult_006 = N2_diff_adult_result[:,4:6][~pd.isnull(N2_diff_adult_result[:,4]),:]
n2_diff_adult_008 = N2_diff_adult_result[:,6:8][~pd.isnull(N2_diff_adult_result[:,6]),:]
n2_diff_adult_010 = N2_diff_adult_result[:,8:][~pd.isnull(N2_diff_adult_result[:,8]),:]

# # JN2389
# jn2389_diff_dauer_1 = JN2389_diff_dauer_result[:,0:2][~pd.isnull(JN2389_diff_dauer_result[:,0]),:]
# jn2389_diff_dauer_2 = JN2389_diff_dauer_result[:,2:4][~pd.isnull(JN2389_diff_dauer_result[:,2]),:]
# jn2389_diff_dauer_4 = JN2389_diff_dauer_result[:,4:6][~pd.isnull(JN2389_diff_dauer_result[:,4]),:]
# jn2389_diff_dauer_6 = JN2389_diff_dauer_result[:,6:8][~pd.isnull(JN2389_diff_dauer_result[:,6]),:]
# jn2389_diff_dauer_8 = JN2389_diff_dauer_result[:,8:][~pd.isnull(JN2389_diff_dauer_result[:,8]),:]
# jn2389_diff_adult_002 = JN2389_diff_adult_result[:,0:2][~pd.isnull(JN2389_diff_adult_result[:,0]),:]
# jn2389_diff_adult_004 = JN2389_diff_adult_result[:,2:4][~pd.isnull(JN2389_diff_adult_result[:,2]),:]
# jn2389_diff_adult_006 = JN2389_diff_adult_result[:,4:6][~pd.isnull(JN2389_diff_adult_result[:,4]),:]
# jn2389_diff_adult_008 = JN2389_diff_adult_result[:,6:8][~pd.isnull(JN2389_diff_adult_result[:,6]),:]
# jn2389_diff_adult_010 = JN2389_diff_adult_result[:,8:][~pd.isnull(JN2389_diff_adult_result[:,8]),:]

# JN1713
jn1713_diff_dauer_1 = JN1713_diff_dauer_result[:,0:2][~pd.isnull(JN1713_diff_dauer_result[:,0]),:]
jn1713_diff_dauer_2 = JN1713_diff_dauer_result[:,2:4][~pd.isnull(JN1713_diff_dauer_result[:,2]),:]
jn1713_diff_dauer_4 = JN1713_diff_dauer_result[:,4:6][~pd.isnull(JN1713_diff_dauer_result[:,4]),:]
jn1713_diff_dauer_6 = JN1713_diff_dauer_result[:,6:8][~pd.isnull(JN1713_diff_dauer_result[:,6]),:]
jn1713_diff_dauer_8 = JN1713_diff_dauer_result[:,8:][~pd.isnull(JN1713_diff_dauer_result[:,8]),:]
jn1713_diff_adult_002 = JN1713_diff_adult_result[:,0:2][~pd.isnull(JN1713_diff_adult_result[:,0]),:]
jn1713_diff_adult_004 = JN1713_diff_adult_result[:,2:4][~pd.isnull(JN1713_diff_adult_result[:,2]),:]
jn1713_diff_adult_006 = JN1713_diff_adult_result[:,4:6][~pd.isnull(JN1713_diff_adult_result[:,4]),:]
jn1713_diff_adult_008 = JN1713_diff_adult_result[:,6:8][~pd.isnull(JN1713_diff_adult_result[:,6]),:]
jn1713_diff_adult_010 = JN1713_diff_adult_result[:,8:][~pd.isnull(JN1713_diff_adult_result[:,8]),:]


In [ ]:
# reversal duration w/ 'back=1' (actual reversed ones)

# N2
n2_diff_dauer_1_dur = n2_diff_dauer_1[n2_diff_dauer_1[:,1]==1,0]  # where 'back=1'
n2_diff_dauer_2_dur = n2_diff_dauer_2[n2_diff_dauer_2[:,1]==1,0]
n2_diff_dauer_4_dur = n2_diff_dauer_4[n2_diff_dauer_4[:,1]==1,0]
n2_diff_dauer_6_dur = n2_diff_dauer_6[n2_diff_dauer_6[:,1]==1,0]
n2_diff_dauer_8_dur = n2_diff_dauer_8[n2_diff_dauer_8[:,1]==1,0]
n2_diff_adult_002_dur = n2_diff_adult_002[n2_diff_adult_002[:,1]==1,0]
n2_diff_adult_004_dur = n2_diff_adult_004[n2_diff_adult_004[:,1]==1,0]
n2_diff_adult_006_dur = n2_diff_adult_006[n2_diff_adult_006[:,1]==1,0]
n2_diff_adult_008_dur = n2_diff_adult_008[n2_diff_adult_008[:,1]==1,0]
n2_diff_adult_010_dur = n2_diff_adult_010[n2_diff_adult_010[:,1]==1,0]

# # JN2389
# jn2389_diff_dauer_1_dur = jn2389_diff_dauer_1[jn2389_diff_dauer_1[:,1]==1,0]
# jn2389_diff_dauer_2_dur = jn2389_diff_dauer_2[jn2389_diff_dauer_2[:,1]==1,0]
# jn2389_diff_dauer_4_dur = jn2389_diff_dauer_4[jn2389_diff_dauer_4[:,1]==1,0]
# jn2389_diff_dauer_6_dur = jn2389_diff_dauer_6[jn2389_diff_dauer_6[:,1]==1,0]
# jn2389_diff_dauer_8_dur = jn2389_diff_dauer_8[jn2389_diff_dauer_8[:,1]==1,0]
# jn2389_diff_adult_002_dur = jn2389_diff_adult_002[jn2389_diff_adult_002[:,1]==1,0]
# jn2389_diff_adult_004_dur = jn2389_diff_adult_004[jn2389_diff_adult_004[:,1]==1,0]
# jn2389_diff_adult_006_dur = jn2389_diff_adult_006[jn2389_diff_adult_006[:,1]==1,0]
# jn2389_diff_adult_008_dur = jn2389_diff_adult_008[jn2389_diff_adult_008[:,1]==1,0]
# jn2389_diff_adult_010_dur = jn2389_diff_adult_010[jn2389_diff_adult_010[:,1]==1,0]

# JN1713
jn1713_diff_dauer_1_dur = jn1713_diff_dauer_1[jn1713_diff_dauer_1[:,1]==1,0]
jn1713_diff_dauer_2_dur = jn1713_diff_dauer_2[jn1713_diff_dauer_2[:,1]==1,0]
jn1713_diff_dauer_4_dur = jn1713_diff_dauer_4[jn1713_diff_dauer_4[:,1]==1,0]
jn1713_diff_dauer_6_dur = jn1713_diff_dauer_6[jn1713_diff_dauer_6[:,1]==1,0]
jn1713_diff_dauer_8_dur = jn1713_diff_dauer_8[jn1713_diff_dauer_8[:,1]==1,0]
jn1713_diff_adult_002_dur = jn1713_diff_adult_002[jn1713_diff_adult_002[:,1]==1,0]
jn1713_diff_adult_004_dur = jn1713_diff_adult_004[jn1713_diff_adult_004[:,1]==1,0]
jn1713_diff_adult_006_dur = jn1713_diff_adult_006[jn1713_diff_adult_006[:,1]==1,0]
jn1713_diff_adult_008_dur = jn1713_diff_adult_008[jn1713_diff_adult_008[:,1]==1,0]
jn1713_diff_adult_010_dur = jn1713_diff_adult_010[jn1713_diff_adult_010[:,1]==1,0]


In [ ]:
# statistic test (N2)

temp0 = np.copy(n2_diff_adult_002_dur)
temp1 = np.copy(n2_diff_adult_004_dur)
temp2 = np.copy(n2_diff_adult_006_dur)
temp3 = np.copy(n2_diff_adult_008_dur)
temp4 = np.copy(n2_diff_adult_010_dur)
temp5 = np.copy(n2_diff_dauer_1_dur)
temp6 = np.copy(n2_diff_dauer_2_dur)
temp7 = np.copy(n2_diff_dauer_4_dur)
temp8 = np.copy(n2_diff_dauer_6_dur)
temp9 = np.copy(n2_diff_dauer_8_dur)

print("Adult :")
print(np.round(stats.ranksums(temp0,temp1)[1],4), " ", np.round(stats.ranksums(temp0,temp2)[1],4), " ",\
      np.round(stats.ranksums(temp0,temp3)[1],4), " ", np.round(stats.ranksums(temp0,temp4)[1],4), " ",\
      np.round(stats.ranksums(temp1,temp2)[1],4), " ", np.round(stats.ranksums(temp1,temp3)[1],4), " ",\
      np.round(stats.ranksums(temp1,temp4)[1],4), " ", np.round(stats.ranksums(temp2,temp3)[1],4), " ",\
      np.round(stats.ranksums(temp2,temp4)[1],4), " ", np.round(stats.ranksums(temp3,temp4)[1],4), "\n")
print("Dauer :")
print(np.round(stats.ranksums(temp5,temp6)[1],4), " ", np.round(stats.ranksums(temp5,temp7)[1],4), " ",\
      np.round(stats.ranksums(temp5,temp8)[1],4), " ", np.round(stats.ranksums(temp5,temp9)[1],4), " ",\
      np.round(stats.ranksums(temp6,temp7)[1],4), " ", np.round(stats.ranksums(temp6,temp8)[1],4), " ",\
      np.round(stats.ranksums(temp6,temp9)[1],4), " ", np.round(stats.ranksums(temp7,temp8)[1],4), " ",\
      np.round(stats.ranksums(temp7,temp9)[1],4), " ", np.round(stats.ranksums(temp8,temp9)[1],4))


In [ ]:
# mean ; error

# from scipy.stats import mannwhitneyu
# from scipy.stats import ranksums

N2A_002 = np.copy(n2_diff_adult_002_dur)
N2A_004 = np.copy(n2_diff_adult_004_dur)
N2A_006 = np.copy(n2_diff_adult_006_dur)
N2A_008 = np.copy(n2_diff_adult_008_dur)
N2A_010 = np.copy(n2_diff_adult_010_dur)

N2D_1 = np.copy(n2_diff_dauer_1_dur)
N2D_2 = np.copy(n2_diff_dauer_2_dur)
N2D_4 = np.copy(n2_diff_dauer_4_dur)
N2D_6 = np.copy(n2_diff_dauer_6_dur)
N2D_8 = np.copy(n2_diff_dauer_8_dur)


# mean
N2A_002_mean = np.mean(N2A_002)
N2A_004_mean = np.mean(N2A_004)
N2A_006_mean = np.mean(N2A_006)
N2A_008_mean = np.mean(N2A_008)
N2A_010_mean = np.mean(N2A_010)

N2D_1_mean = np.mean(N2D_1)
N2D_2_mean = np.mean(N2D_2)
N2D_4_mean = np.mean(N2D_4)
N2D_6_mean = np.mean(N2D_6)
N2D_8_mean = np.mean(N2D_8)


# sem
N2A_002_sem = stats.sem(N2A_002)
N2A_004_sem = stats.sem(N2A_004)
N2A_006_sem = stats.sem(N2A_006)
N2A_008_sem = stats.sem(N2A_008)
N2A_010_sem = stats.sem(N2A_010)

N2D_1_sem = stats.sem(N2D_1)
N2D_2_sem = stats.sem(N2D_2)
N2D_4_sem = stats.sem(N2D_4)
N2D_6_sem = stats.sem(N2D_6)
N2D_8_sem = stats.sem(N2D_8)


In [ ]:
# bar graph (adult)

b_width = 0.6; v_width = 0.6; ssz=14; fsz=14


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))

xpos = [1,2,3,4,5]

ax1.set_ylabel('number of sine waves', fontsize=14, fontname='sans-serif')
ax1.set_title('Adult', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], N2A_002_mean, yerr=N2A_002_sem, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[1], N2A_004_mean, yerr=N2A_004_sem, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[2], N2A_006_mean, yerr=N2A_006_sem, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[3], N2A_008_mean, yerr=N2A_008_sem, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[4], N2A_010_mean, yerr=N2A_010_sem, width=b_width, color="#1F77B4", alpha=0.7)

ax1.set_xlim(0.4,5.6)
ax1.set_ylim(0,2.7)
label = ["0.02","0.04","0.06","0.08","0.10"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=12.5, rotation=0, fontname='sans-serif')
ax1.set_yticks(np.arange(0,3.0,0.5))
ax1.set_yticklabels(["0","0.5","1.0","1.5","2.0","2.5"], fontsize=12.5, rotation=0, fontname='sans-serif')

ax1.plot([0.8,5.2], [2.32,2.32], lw=0.8, c='k')
ax1.text(3.0, 2.38, "n.s", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')

plt.show()


In [ ]:
# bar graph (dauer)

b_width = 0.6; v_width = 0.6; ssz=14; fsz=14


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))

xpos = [1,2,3,4,5]

ax1.set_ylabel('number of sine waves', fontsize=14, fontname='sans-serif')
ax1.set_title('Dauer', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], N2D_1_mean, yerr=N2D_1_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.bar(xpos[1], N2D_2_mean, yerr=N2D_2_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.bar(xpos[2], N2D_4_mean, yerr=N2D_4_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.bar(xpos[3], N2D_6_mean, yerr=N2D_6_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.bar(xpos[4], N2D_8_mean, yerr=N2D_8_sem, width=b_width, color="#FF7F0E", alpha=0.7)

ax1.set_xlim(0.4,5.6)
ax1.set_ylim(0,2.7)
label = ["1.0","2.0","4.0","6.0","8.0"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=12.5, rotation=0, fontname='sans-serif')
ax1.set_yticks(np.arange(0,3.0,0.5))
ax1.set_yticklabels(["0","0.5","1.0","1.5","2.0","2.5"], fontsize=12.5, rotation=0, fontname='sans-serif')

ax1.plot([0.8,5.2], [1.40,1.40], lw=0.8, c='k')
ax1.text(3.0, 1.46, "n.s", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')

plt.show()


### ASH - AVD Cx36 line
- YA ~ 1 day adults
- 5 sec. optogenetic stimuli (green; ChR2)

In [ ]:
# read file

ASH_AVD_file = r'your_directory\ASH-AVD_Cx36_nociception_test.xlsx'

ASH_AVD_df = pd.read_excel(ASH_AVD_file, sheet_name='240901', header=None)
ASH_AVD_np = np.array(ASH_AVD_df)
ASH_AVD_o_result = np.copy(ASH_AVD_np[2:,0])  # adult; w/  Cx36
ASH_AVD_x_result = np.copy(ASH_AVD_np[2:,1])  # adult; w/o Cx36


In [ ]:
# read file

noci_file = r'your_directory\nociception_test.xlsx'

noci_df = pd.read_excel(noci_file, sheet_name='JN2389_ATR_o', header=None)
noci_np = np.array(noci_df)
jn2389_result = noci_np[4:,0:16]  # dauer=col0-7 // adult=col10-15 (1 experiment set == 2 columns)


In [ ]:
# reversal duration - NaN cutoff

# JN2389 dauer
jn2389_dauer_1 = jn2389_result[:,0:2][~pd.isnull(jn2389_result[:,0]),:]  # dauer set 1 (col 0,1)
jn2389_dauer_1_dur = jn2389_dauer_1[:,0]  # all
# jn2389_dauer_1_dur = jn2389_dauer_1[jn2389_dauer_1[:,1]==1,0]  # where 'back=1'

# ASH-AVD
ASH_AVD_x = ASH_AVD_x_result[~pd.isnull(ASH_AVD_x_result)]
ASH_AVD_o = ASH_AVD_o_result[~pd.isnull(ASH_AVD_o_result)]


In [ ]:
jn2389_dauer_1_dur[jn2389_dauer_1_dur!=0]

In [ ]:
# statistic test (Cx36)

temp0 = np.copy(ASH_AVD_x[ASH_AVD_x!=0]).astype(np.float64)
temp1 = np.copy(ASH_AVD_o[ASH_AVD_o!=0]).astype(np.float64)
temp2 = np.copy(jn2389_dauer_1_dur[jn2389_dauer_1_dur!=0]).astype(np.float64)

print("Cx36 O vs. Cx36 X : ", stats.mannwhitneyu(temp0,temp1, method='exact')[1])
print("Cx36 O vs. Dauer  : ", stats.mannwhitneyu(temp1,temp2, method='exact')[1])


In [ ]:
temp0 = np.copy(ASH_AVD_x[ASH_AVD_x!=0]).astype(np.float64)
temp1 = np.copy(ASH_AVD_o[ASH_AVD_o!=0]).astype(np.float64)
temp2 = np.copy(jn2389_dauer_1_dur[jn2389_dauer_1_dur!=0]).astype(np.float64)

print(temp0.shape)
print(temp1.shape)
print(temp2.shape)

In [ ]:
# mean ; error

# from scipy.stats import mannwhitneyu
# from scipy.stats import ranksums

ASH_AVD_Rwave_adult = np.copy(ASH_AVD_x[ASH_AVD_x!=0])
ASH_AVD_Rwave_cx36 = np.copy(ASH_AVD_o[ASH_AVD_o!=0])
ASH_AVD_Rwave_dauer = np.copy(jn2389_dauer_1_dur[jn2389_dauer_1_dur!=0])


ASH_AVD_Rwave_adult_mean = np.mean(ASH_AVD_Rwave_adult)
ASH_AVD_Rwave_cx36_mean = np.mean(ASH_AVD_Rwave_cx36)
ASH_AVD_Rwave_dauer_mean = np.mean(ASH_AVD_Rwave_dauer)

ASH_AVD_Rwave_adult_sem = stats.sem(ASH_AVD_Rwave_adult)
ASH_AVD_Rwave_cx36_sem = stats.sem(ASH_AVD_Rwave_cx36)
ASH_AVD_Rwave_dauer_sem = stats.sem(ASH_AVD_Rwave_dauer)


In [ ]:
# bar graph


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3.2,4))

xpos = [1,2,3]

ax1.set_ylabel('number of sine waves', fontsize=14, fontname='sans-serif')
ax1.bar(xpos[0], ASH_AVD_Rwave_adult_mean, yerr=ASH_AVD_Rwave_adult_sem, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[1], ASH_AVD_Rwave_cx36_mean, yerr=ASH_AVD_Rwave_cx36_sem, width=b_width, color="#8C564B", alpha=0.7)
ax1.bar(xpos[2], ASH_AVD_Rwave_dauer_mean, yerr=ASH_AVD_Rwave_dauer_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,3.6)
ax1.set_ylim(0,2.38)
label = ["Cx36 X\nAdult","Cx36 O\nAdult","\nDauer"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=12.5, rotation=0, fontname='sans-serif')
ax1.set_yticks(np.arange(0,2.5,0.5))
ax1.set_yticklabels(["0","0.5","1.0","1.5","2.0"], fontsize=12.5, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [2.08,2.08], lw=0.8, c='k')
ax1.text(1.5, 2.08, "***", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
ax1.plot([1.9,3.1], [1.4,1.4], lw=0.8, c='k')
ax1.text(2.5, 1.43, "n.s", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')

plt.show()


### AVB - AVD Cx36 line
- YA ~ 1 day adults
- 5 sec. optogenetic stimuli (green; ChR2)

In [ ]:
# read file

AVB_AVD_file = r'your_directory\AVB-AVD_Cx36_nociception_test.xlsx'

AVB_AVD_df = pd.read_excel(AVB_AVD_file, sheet_name='250721', header=None)
AVB_AVD_np = np.array(AVB_AVD_df)
AVB_AVD_o_result = np.copy(AVB_AVD_np[2:,0])  # adult; w/  Cx36
AVB_AVD_x_result = np.copy(AVB_AVD_np[2:,1])  # adult; w/o Cx36


In [ ]:
# read file

noci_file = r'your_directory\nociception_test.xlsx'

noci_df = pd.read_excel(noci_file, sheet_name='JN2389_ATR_o', header=None)
noci_np = np.array(noci_df)
jn2389_result = noci_np[4:,0:16]  # dauer=col0-7 // adult=col10-15 (1 experiment set == 2 columns)


In [ ]:
# reversal duration - NaN cutoff

# JN2389 dauer
jn2389_dauer_2 = jn2389_result[:,2:4][~pd.isnull(jn2389_result[:,2]),:]  # dauer set 2 (col 2,3)
jn2389_dauer_2_dur = jn2389_dauer_2[:,0]  # all
# jn2389_dauer_2_dur = jn2389_dauer_2[jn2389_dauer_2[:,1]==1,0]  # where 'back=1'

# AVB-AVD
AVB_AVD_x = AVB_AVD_x_result[~pd.isnull(AVB_AVD_x_result)]
AVB_AVD_o = AVB_AVD_o_result[~pd.isnull(AVB_AVD_o_result)]


In [ ]:
# statistic test (Cx36)

temp0 = np.copy(AVB_AVD_x[AVB_AVD_x!=0]).astype(np.float64)
temp1 = np.copy(AVB_AVD_o[AVB_AVD_o!=0]).astype(np.float64)
temp2 = np.copy(jn2389_dauer_2_dur[jn2389_dauer_2_dur!=0]).astype(np.float64)

print("Cx36 O vs. Cx36 X : ", stats.mannwhitneyu(temp0,temp1, method='exact')[1])
print("Cx36 O vs. Dauer  : ", stats.mannwhitneyu(temp1,temp2, method='exact')[1])


In [ ]:
temp0 = np.copy(AVB_AVD_x[AVB_AVD_x!=0]).astype(np.float64)
temp1 = np.copy(AVB_AVD_o[AVB_AVD_o!=0]).astype(np.float64)
temp2 = np.copy(jn2389_dauer_2_dur[jn2389_dauer_2_dur!=0]).astype(np.float64)

print(temp0.shape)
print(temp1.shape)
print(temp2.shape)

In [ ]:
# mean ; error

# from scipy.stats import mannwhitneyu
# from scipy.stats import ranksums

AVB_AVD_Rwave_adult = np.copy(AVB_AVD_x[AVB_AVD_x!=0])
AVB_AVD_Rwave_cx36 = np.copy(AVB_AVD_o[AVB_AVD_o!=0])
AVB_AVD_Rwave_dauer = np.copy(jn2389_dauer_2_dur[jn2389_dauer_2_dur!=0])


AVB_AVD_Rwave_adult_mean = np.mean(AVB_AVD_Rwave_adult)
AVB_AVD_Rwave_cx36_mean = np.mean(AVB_AVD_Rwave_cx36)
AVB_AVD_Rwave_dauer_mean = np.mean(AVB_AVD_Rwave_dauer)

AVB_AVD_Rwave_adult_sem = stats.sem(AVB_AVD_Rwave_adult)
AVB_AVD_Rwave_cx36_sem = stats.sem(AVB_AVD_Rwave_cx36)
AVB_AVD_Rwave_dauer_sem = stats.sem(AVB_AVD_Rwave_dauer)


In [ ]:
# bar graph


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3.2,4))

xpos = [1,2,3]

ax1.set_ylabel('number of sine waves', fontsize=14, fontname='sans-serif')
ax1.bar(xpos[0], AVB_AVD_Rwave_adult_mean, yerr=AVB_AVD_Rwave_adult_sem, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[1], AVB_AVD_Rwave_cx36_mean, yerr=AVB_AVD_Rwave_cx36_sem, width=b_width, color="#8C564B", alpha=0.7)
ax1.bar(xpos[2], AVB_AVD_Rwave_dauer_mean, yerr=AVB_AVD_Rwave_dauer_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,3.6)
ax1.set_ylim(0,2.1)
label = ["Cx36 X\nAdult","Cx36 O\nAdult","\nDauer"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=12.5, rotation=0, fontname='sans-serif')
ax1.set_yticks(np.arange(0,2.5,0.5))
ax1.set_yticklabels(["0","0.5","1.0","1.5","2.0"], fontsize=12.5, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [1.86,1.86], lw=0.8, c='k')
ax1.text(1.5, 1.86, "***", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
ax1.plot([1.9,3.1], [1.32,1.32], lw=0.8, c='k')
ax1.text(2.5, 1.35, "n.s", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')

plt.show()


### ASH - AVA Cx36 line
- YA ~ 1 day adults
- 5 sec. optogenetic stimuli (green; ChR2)

In [ ]:
# read file

ASH_AVA_file = r'your_directory\ASH-AVA_Cx36_nociception_test.xlsx'

ASH_AVA_df = pd.read_excel(ASH_AVA_file, sheet_name='250930', header=None)
ASH_AVA_np = np.array(ASH_AVA_df)
ASH_AVA_o_result = np.copy(ASH_AVA_np[2:,0])  # adult; w/  Cx36
ASH_AVA_x_result = np.copy(ASH_AVA_np[2:,1])  # adult; w/o Cx36


In [ ]:
# read file

noci_file = r'your_directory\nociception_test.xlsx'

noci_df = pd.read_excel(noci_file, sheet_name='JN2389_ATR_o', header=None)
noci_np = np.array(noci_df)
jn2389_result = noci_np[4:,0:16]  # dauer=col0-7 // adult=col10-15 (1 experiment set == 2 columns)


In [ ]:
# reversal duration - NaN cutoff

# JN2389 dauer
jn2389_dauer_3 = jn2389_result[:,4:6][~pd.isnull(jn2389_result[:,4]),:]  # dauer set 3 (col 4,5)
jn2389_dauer_3_dur = jn2389_dauer_3[:,0]  # all
# jn2389_dauer_3_dur = jn2389_dauer_3[jn2389_dauer_3[:,1]==1,0]  # where 'back=1'

# ASH-AVA
ASH_AVA_x = ASH_AVA_x_result[~pd.isnull(ASH_AVA_x_result)]
ASH_AVA_o = ASH_AVA_o_result[~pd.isnull(ASH_AVA_o_result)]


In [ ]:
# statistic test (Cx36)

temp0 = np.copy(ASH_AVA_x[ASH_AVA_x!=0]).astype(np.float64)
temp1 = np.copy(ASH_AVA_o[ASH_AVA_o!=0]).astype(np.float64)
temp2 = np.copy(jn2389_dauer_3_dur[jn2389_dauer_3_dur!=0]).astype(np.float64)

print("Cx36 O vs. Cx36 X : ", stats.mannwhitneyu(temp0,temp1, method='exact')[1])
print("Cx36 O vs. Dauer  : ", stats.mannwhitneyu(temp1,temp2, method='exact')[1])


In [ ]:
temp0 = np.copy(ASH_AVA_x[ASH_AVA_x!=0]).astype(np.float64)
temp1 = np.copy(ASH_AVA_o[ASH_AVA_o!=0]).astype(np.float64)
temp2 = np.copy(jn2389_dauer_3_dur[jn2389_dauer_3_dur!=0]).astype(np.float64)

print(temp0.shape)
print(temp1.shape)
print(temp2.shape)

In [ ]:
# mean ; error

# from scipy.stats import mannwhitneyu
# from scipy.stats import ranksums

ASH_AVA_Rwave_adult = np.copy(ASH_AVA_x[ASH_AVA_x!=0])
ASH_AVA_Rwave_cx36 = np.copy(ASH_AVA_o[ASH_AVA_o!=0])
ASH_AVA_Rwave_dauer = np.copy(jn2389_dauer_3_dur[jn2389_dauer_3_dur!=0])


ASH_AVA_Rwave_adult_mean = np.mean(ASH_AVA_Rwave_adult)
ASH_AVA_Rwave_cx36_mean = np.mean(ASH_AVA_Rwave_cx36)
ASH_AVA_Rwave_dauer_mean = np.mean(ASH_AVA_Rwave_dauer)

ASH_AVA_Rwave_adult_sem = stats.sem(ASH_AVA_Rwave_adult)
ASH_AVA_Rwave_cx36_sem = stats.sem(ASH_AVA_Rwave_cx36)
ASH_AVA_Rwave_dauer_sem = stats.sem(ASH_AVA_Rwave_dauer)


In [ ]:
# bar graph


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3.2,4))

xpos = [1,2,3]

ax1.set_ylabel('number of sine waves', fontsize=14, fontname='sans-serif')
ax1.bar(xpos[0], ASH_AVA_Rwave_adult_mean, yerr=ASH_AVA_Rwave_adult_sem, width=b_width, color="#1F77B4", alpha=0.7)
ax1.bar(xpos[1], ASH_AVA_Rwave_cx36_mean, yerr=ASH_AVA_Rwave_cx36_sem, width=b_width, color="#8C564B", alpha=0.7)
ax1.bar(xpos[2], ASH_AVA_Rwave_dauer_mean, yerr=ASH_AVA_Rwave_dauer_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,3.6)
ax1.set_ylim(0,2.38)
label = ["Cx36 X\nAdult","Cx36 O\nAdult","\nDauer"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=12.5, rotation=0, fontname='sans-serif')
ax1.set_yticks(np.arange(0,2.5,0.5))
ax1.set_yticklabels(["0","0.5","1.0","1.5","2.0"], fontsize=12.5, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [2.09,2.09], lw=0.8, c='k')
ax1.text(1.5, 2.14, "n.s", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
ax1.plot([1.9,3.1], [1.85,1.85], lw=0.8, c='k')
ax1.text(2.5, 1.85, "***", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')

plt.show()


## Reproducing `Avoidance Index` from Pechuk *et al*.
- Tail-drop assay<br>
- 4-sec time window

In [ ]:
# read file

AI_file = r'your_directory\avoidance_index_test_taildrop.xlsx'

## dauer
AI_d01_df = pd.read_excel(AI_file, sheet_name='dauer_0.1', header=None)
AI_d01_np = np.array(AI_d01_df)[1:,1:]; print(AI_d01_np.shape)  # has to be 10-by-N shape (10 = trial; N = # 0f worms)
AI_d02_df = pd.read_excel(AI_file, sheet_name='dauer_0.2', header=None)
AI_d02_np = np.array(AI_d02_df)[1:,1:]; print(AI_d02_np.shape)
AI_d04_df = pd.read_excel(AI_file, sheet_name='dauer_0.4', header=None)
AI_d04_np = np.array(AI_d04_df)[1:,1:]; print(AI_d04_np.shape)
AI_d06_df = pd.read_excel(AI_file, sheet_name='dauer_0.6', header=None)
AI_d06_np = np.array(AI_d06_df)[1:,1:]; print(AI_d06_np.shape)
AI_d08_df = pd.read_excel(AI_file, sheet_name='dauer_0.8', header=None)
AI_d08_np = np.array(AI_d08_df)[1:,1:]; print(AI_d08_np.shape)
AI_d10_df = pd.read_excel(AI_file, sheet_name='dauer_1.0', header=None)
AI_d10_np = np.array(AI_d10_df)[1:,1:]; print(AI_d10_np.shape)
AI_d12_df = pd.read_excel(AI_file, sheet_name='dauer_1.2', header=None)
AI_d12_np = np.array(AI_d12_df)[1:,1:]; print(AI_d12_np.shape)
AI_d14_df = pd.read_excel(AI_file, sheet_name='dauer_1.4', header=None)
AI_d14_np = np.array(AI_d14_df)[1:,1:]; print(AI_d14_np.shape)
AI_d16_df = pd.read_excel(AI_file, sheet_name='dauer_1.6', header=None)
AI_d16_np = np.array(AI_d16_df)[1:,1:]; print(AI_d16_np.shape)
AI_d18_df = pd.read_excel(AI_file, sheet_name='dauer_1.8', header=None)
AI_d18_np = np.array(AI_d18_df)[1:,1:]; print(AI_d18_np.shape)
AI_d22_df = pd.read_excel(AI_file, sheet_name='dauer_2.2', header=None)
AI_d22_np = np.array(AI_d22_df)[1:,1:]; print(AI_d22_np.shape)
AI_d26_df = pd.read_excel(AI_file, sheet_name='dauer_2.6', header=None)
AI_d26_np = np.array(AI_d26_df)[1:,1:]; print(AI_d26_np.shape)
AI_d30_df = pd.read_excel(AI_file, sheet_name='dauer_3.0', header=None)
AI_d30_np = np.array(AI_d30_df)[1:,1:]; print(AI_d30_np.shape)
AI_d36_df = pd.read_excel(AI_file, sheet_name='dauer_3.6', header=None)
AI_d36_np = np.array(AI_d36_df)[1:,1:]; print(AI_d36_np.shape, "\n")

## adult
AI_a002_df = pd.read_excel(AI_file, sheet_name='adult_0.02', header=None)
AI_a002_np = np.array(AI_a002_df)[1:,1:]; print(AI_a002_np.shape)
AI_a004_df = pd.read_excel(AI_file, sheet_name='adult_0.04', header=None)
AI_a004_np = np.array(AI_a004_df)[1:,1:]; print(AI_a004_np.shape)
AI_a006_df = pd.read_excel(AI_file, sheet_name='adult_0.06', header=None)
AI_a006_np = np.array(AI_a006_df)[1:,1:]; print(AI_a006_np.shape)
AI_a008_df = pd.read_excel(AI_file, sheet_name='adult_0.08', header=None)
AI_a008_np = np.array(AI_a008_df)[1:,1:]; print(AI_a008_np.shape)
AI_a010_df = pd.read_excel(AI_file, sheet_name='adult_0.10', header=None)
AI_a010_np = np.array(AI_a010_df)[1:,1:]; print(AI_a010_np.shape)


In [ ]:
# get avoidance index

## dauer
AI_d01_val = np.sum(AI_d01_np, axis=0) / AI_d01_np.shape[0]
AI_d02_val = np.sum(AI_d02_np, axis=0) / AI_d02_np.shape[0]
AI_d04_val = np.sum(AI_d04_np, axis=0) / AI_d04_np.shape[0]
AI_d06_val = np.sum(AI_d06_np, axis=0) / AI_d06_np.shape[0]
AI_d08_val = np.sum(AI_d08_np, axis=0) / AI_d08_np.shape[0]
AI_d10_val = np.sum(AI_d10_np, axis=0) / AI_d10_np.shape[0]
AI_d12_val = np.sum(AI_d12_np, axis=0) / AI_d12_np.shape[0]
AI_d14_val = np.sum(AI_d14_np, axis=0) / AI_d14_np.shape[0]
AI_d16_val = np.sum(AI_d16_np, axis=0) / AI_d16_np.shape[0]
AI_d18_val = np.sum(AI_d18_np, axis=0) / AI_d18_np.shape[0]
AI_d22_val = np.sum(AI_d22_np, axis=0) / AI_d22_np.shape[0]
AI_d26_val = np.sum(AI_d26_np, axis=0) / AI_d26_np.shape[0]
AI_d30_val = np.sum(AI_d30_np, axis=0) / AI_d30_np.shape[0]
AI_d36_val = np.sum(AI_d36_np, axis=0) / AI_d36_np.shape[0]

## adult
AI_a002_val = np.sum(AI_a002_np, axis=0) / AI_a002_np.shape[0]
AI_a004_val = np.sum(AI_a004_np, axis=0) / AI_a004_np.shape[0]
AI_a006_val = np.sum(AI_a006_np, axis=0) / AI_a006_np.shape[0]
AI_a008_val = np.sum(AI_a008_np, axis=0) / AI_a008_np.shape[0]
AI_a010_val = np.sum(AI_a010_np, axis=0) / AI_a010_np.shape[0]


In [ ]:
# mean & error (standard error of the mean)

## dauer
AI_d01_mean = np.mean(AI_d01_val)
AI_d02_mean = np.mean(AI_d02_val)
AI_d04_mean = np.mean(AI_d04_val)
AI_d06_mean = np.mean(AI_d06_val)
AI_d08_mean = np.mean(AI_d08_val)
AI_d10_mean = np.mean(AI_d10_val)
AI_d12_mean = np.mean(AI_d12_val)
AI_d14_mean = np.mean(AI_d14_val)
AI_d16_mean = np.mean(AI_d16_val)
AI_d18_mean = np.mean(AI_d18_val)
AI_d22_mean = np.mean(AI_d22_val)
AI_d26_mean = np.mean(AI_d26_val)
AI_d30_mean = np.mean(AI_d30_val)
AI_d36_mean = np.mean(AI_d36_val)

## adult
AI_a002_mean = np.mean(AI_a002_val)
AI_a004_mean = np.mean(AI_a004_val)
AI_a006_mean = np.mean(AI_a006_val)
AI_a008_mean = np.mean(AI_a008_val)
AI_a010_mean = np.mean(AI_a010_val)


## dauer
AI_d01_sem = stats.sem(AI_d01_val)
AI_d02_sem = stats.sem(AI_d02_val)
AI_d04_sem = stats.sem(AI_d04_val)
AI_d06_sem = stats.sem(AI_d06_val)
AI_d08_sem = stats.sem(AI_d08_val)
AI_d10_sem = stats.sem(AI_d10_val)
AI_d12_sem = stats.sem(AI_d12_val)
AI_d14_sem = stats.sem(AI_d14_val)
AI_d16_sem = stats.sem(AI_d16_val)
AI_d18_sem = stats.sem(AI_d18_val)
AI_d22_sem = stats.sem(AI_d22_val)
AI_d26_sem = stats.sem(AI_d26_val)
AI_d30_sem = stats.sem(AI_d30_val)
AI_d36_sem = stats.sem(AI_d36_val)

## adult
AI_a002_sem = stats.sem(AI_a002_val)
AI_a004_sem = stats.sem(AI_a004_val)
AI_a006_sem = stats.sem(AI_a006_val)
AI_a008_sem = stats.sem(AI_a008_val)
AI_a010_sem = stats.sem(AI_a010_val)


In [ ]:
# plot

fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=True, figsize=(4.5,3))
xpos = np.array([0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8])  # changed for visualization

yvals = [AI_d02_mean, AI_d04_mean, AI_d06_mean, AI_d08_mean,
         AI_d10_mean, AI_d12_mean, AI_d14_mean, AI_d16_mean, AI_d18_mean]
yerrs = [AI_d02_sem, AI_d04_sem, AI_d06_sem, AI_d08_sem,
         AI_d10_sem, AI_d12_sem, AI_d14_sem, AI_d16_sem, AI_d18_sem]


ax1.errorbar(xpos, yvals, yerr=yerrs, c="tab:orange", lw=2, ls='solid', capsize=3, elinewidth=2, label="Dauer") 
ax1.set_xlim(0.1,1.9)
ax1.set_ylim(0,1.05)

ax1.set_xticks(xpos)
ax1.set_xticklabels(["0.2","0.4","0.6","0.8","1.0","1.2","1.4","1.6","1.8"],
                    fontsize=12, rotation=45, fontname='Arial')
ax1.set_xlabel("SDS (%)", fontsize=16, rotation=0, fontname='Arial')

ax1.set_yticks(np.arange(0,1.1,0.2))
ax1.set_yticklabels(["0.0","0.2","0.4","0.6","0.8","1.0"], fontsize=14, rotation=0, fontname='Arial')
ax1.set_ylabel("Avoidance index", fontsize=16, rotation=90, fontname='Arial')

ax1.legend(bbox_to_anchor=(0, 1), loc='upper left', fontsize=12.5)
ax1.spines[['right','top']].set_visible(False)
fig.tight_layout(pad=0.5)

plt.show()


#### Plot y-range matched curve

- Dauer 0.6 -> Adult 0.02 <br><br>
- Dauer 1.6 -> Adult 0.04

In [ ]:
# plot


fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=True, figsize=(4.4,4))
xpos1 = np.arange(1,6,1)
xpos2 = np.array([1.00, 1.27, 1.53, 1.80, 2.06, 2.32, 3.66, 5.00])

# adult
yvals_a = [AI_a002_mean, AI_a004_mean, AI_a006_mean, AI_a008_mean, AI_a010_mean]
yerrs_a = [AI_a002_sem, AI_a004_sem, AI_a006_sem, AI_a008_sem, AI_a010_sem]

ax1.errorbar(xpos1, yvals_a, yerr=yerrs_a, c="#1F77B4", lw=2.0, ls='solid', capsize=2.0, elinewidth=2.0, label="Wild-type adults") 
ax1.set_xlim(0.8,5.2)
ax1.set_ylim(0.1,1.05)

ax1.set_xticks(xpos1)
ax1.set_xticklabels(["0.02","0.04","0.06","0.08","0.10"],
                    fontsize=12.5, rotation=45, fontname='sans-serif')
ax1.set_xlabel("SDS (%) Adult", fontsize=14, rotation=0, fontname='sans-serif')

ax1.set_yticks(np.arange(0.2,1.1,0.2))
ax1.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"], fontsize=12.5, rotation=0, fontname='sans-serif')
ax1.set_ylabel("Avoidance index", fontsize=14, rotation=90, fontname='sans-serif')

ax1.spines[['right']].set_visible(False)

# dauer
ax2 = ax1.secondary_xaxis('top')
yvals_d = [AI_d06_mean, AI_d08_mean, AI_d10_mean, AI_d12_mean,
           AI_d14_mean, AI_d16_mean, AI_d26_mean, AI_d36_mean]
yerrs_d = [AI_d06_sem, AI_d08_sem, AI_d10_sem, AI_d12_sem,
           AI_d14_sem, AI_d16_sem, AI_d26_sem, AI_d36_sem]

ax1.errorbar(xpos2, yvals_d, yerr=yerrs_d, c="#FF7F0E", lw=2.0, ls='solid', capsize=2.0, elinewidth=2.0, label="Wild-type dauers") 
ax2.set_xlim(0.1,1.9)
ax2.set_ylim(0,1.05)

ax2.set_xticks(xpos2)
ax2.set_xticklabels(["0.6","0.8","1.0","1.2","1.4","1.6","2.6","3.6"],
                    fontsize=12.5, rotation=45, fontname='sans-serif')
ax2.set_xlabel("SDS (%) Dauer", fontsize=14, rotation=0, fontname='sans-serif')

ax1.legend(bbox_to_anchor=(1, 0), loc='lower right', fontsize=12.5)
fig.tight_layout(pad=0.5)

plt.show()


## Avoidance Index of `Adult` and `Dauer` using optogenetics
- Different light strength<br><br>
- x30 (on screen)<br><br>
- TL CT 2201 K, TL CCIC 5%

In [ ]:
# read file

AI_opto_file = r'your_directory\avoidance_index_test_optogenetic_range.xlsx'

## dauer
AI_opto_d15_df = pd.read_excel(AI_opto_file, sheet_name='Dauer_15', header=None)
AI_opto_d15_np = np.array(AI_opto_d15_df)[1:,1:]; print(AI_opto_d15_np.shape)  # has to be 10-by-N shape (10 = trial; N = # 0f worms)
AI_opto_d20_df = pd.read_excel(AI_opto_file, sheet_name='Dauer_20', header=None)
AI_opto_d20_np = np.array(AI_opto_d20_df)[1:,1:]; print(AI_opto_d20_np.shape)
AI_opto_d40_df = pd.read_excel(AI_opto_file, sheet_name='Dauer_40', header=None)
AI_opto_d40_np = np.array(AI_opto_d40_df)[1:,1:]; print(AI_opto_d40_np.shape)
AI_opto_d60_df = pd.read_excel(AI_opto_file, sheet_name='Dauer_60', header=None)
AI_opto_d60_np = np.array(AI_opto_d60_df)[1:,1:]; print(AI_opto_d60_np.shape)
AI_opto_d80_df = pd.read_excel(AI_opto_file, sheet_name='Dauer_80', header=None)
AI_opto_d80_np = np.array(AI_opto_d80_df)[1:,1:]; print(AI_opto_d80_np.shape)
AI_opto_d100_df = pd.read_excel(AI_opto_file, sheet_name='Dauer_100', header=None)
AI_opto_d100_np = np.array(AI_opto_d100_df)[1:,1:]; print(AI_opto_d100_np.shape, "\n")

## adult
AI_opto_a15_df = pd.read_excel(AI_opto_file, sheet_name='Adult_15', header=None)
AI_opto_a15_np = np.array(AI_opto_a15_df)[1:,1:]; print(AI_opto_a15_np.shape)
AI_opto_a20_df = pd.read_excel(AI_opto_file, sheet_name='Adult_20', header=None)
AI_opto_a20_np = np.array(AI_opto_a20_df)[1:,1:]; print(AI_opto_a20_np.shape)
AI_opto_a40_df = pd.read_excel(AI_opto_file, sheet_name='Adult_40', header=None)
AI_opto_a40_np = np.array(AI_opto_a40_df)[1:,1:]; print(AI_opto_a40_np.shape)
AI_opto_a60_df = pd.read_excel(AI_opto_file, sheet_name='Adult_60', header=None)
AI_opto_a60_np = np.array(AI_opto_a60_df)[1:,1:]; print(AI_opto_a60_np.shape)
AI_opto_a80_df = pd.read_excel(AI_opto_file, sheet_name='Adult_80', header=None)
AI_opto_a80_np = np.array(AI_opto_a80_df)[1:,1:]; print(AI_opto_a80_np.shape)
AI_opto_a100_df = pd.read_excel(AI_opto_file, sheet_name='Adult_100', header=None)
AI_opto_a100_np = np.array(AI_opto_a100_df)[1:,1:]; print(AI_opto_a100_np.shape)


In [ ]:
# get avoidance index

## dauer
AI_opto_d15_val = np.sum(AI_opto_d15_np, axis=0) / AI_opto_d15_np.shape[0]
AI_opto_d20_val = np.sum(AI_opto_d20_np, axis=0) / AI_opto_d20_np.shape[0]
AI_opto_d40_val = np.sum(AI_opto_d40_np, axis=0) / AI_opto_d40_np.shape[0]
AI_opto_d60_val = np.sum(AI_opto_d60_np, axis=0) / AI_opto_d60_np.shape[0]
AI_opto_d80_val = np.sum(AI_opto_d80_np, axis=0) / AI_opto_d80_np.shape[0]
AI_opto_d100_val = np.sum(AI_opto_d100_np, axis=0) / AI_opto_d100_np.shape[0]

## adult
AI_opto_a15_val = np.sum(AI_opto_a15_np, axis=0) / AI_opto_a15_np.shape[0]
AI_opto_a20_val = np.sum(AI_opto_a20_np, axis=0) / AI_opto_a20_np.shape[0]
AI_opto_a40_val = np.sum(AI_opto_a40_np, axis=0) / AI_opto_a40_np.shape[0]
AI_opto_a60_val = np.sum(AI_opto_a60_np, axis=0) / AI_opto_a60_np.shape[0]
AI_opto_a80_val = np.sum(AI_opto_a80_np, axis=0) / AI_opto_a80_np.shape[0]
AI_opto_a100_val = np.sum(AI_opto_a100_np, axis=0) / AI_opto_a100_np.shape[0]


In [ ]:
# mean & error (standard error of the mean)

## dauer
AI_opto_d15_mean = np.mean(AI_opto_d15_val)
AI_opto_d20_mean = np.mean(AI_opto_d20_val)
AI_opto_d40_mean = np.mean(AI_opto_d40_val)
AI_opto_d60_mean = np.mean(AI_opto_d60_val)
AI_opto_d80_mean = np.mean(AI_opto_d80_val)
AI_opto_d100_mean = np.mean(AI_opto_d100_val)

## adult
AI_opto_a15_mean = np.mean(AI_opto_a15_val)
AI_opto_a20_mean = np.mean(AI_opto_a20_val)
AI_opto_a40_mean = np.mean(AI_opto_a40_val)
AI_opto_a60_mean = np.mean(AI_opto_a60_val)
AI_opto_a80_mean = np.mean(AI_opto_a80_val)
AI_opto_a100_mean = np.mean(AI_opto_a100_val)


## dauer
AI_opto_d15_sem = stats.sem(AI_opto_d15_val)
AI_opto_d20_sem = stats.sem(AI_opto_d20_val)
AI_opto_d40_sem = stats.sem(AI_opto_d40_val)
AI_opto_d60_sem = stats.sem(AI_opto_d60_val)
AI_opto_d80_sem = stats.sem(AI_opto_d80_val)
AI_opto_d100_sem = stats.sem(AI_opto_d100_val)

## adult
AI_opto_a15_sem = stats.sem(AI_opto_a15_val)
AI_opto_a20_sem = stats.sem(AI_opto_a20_val)
AI_opto_a40_sem = stats.sem(AI_opto_a40_val)
AI_opto_a60_sem = stats.sem(AI_opto_a60_val)
AI_opto_a80_sem = stats.sem(AI_opto_a80_val)
AI_opto_a100_sem = stats.sem(AI_opto_a100_val)


In [ ]:
# statistical test

print(" 15 : ", stats.mannwhitneyu(AI_opto_d15_val,  AI_opto_a15_val,  method='auto')[1])
print(" 20 : ", stats.mannwhitneyu(AI_opto_d20_val,  AI_opto_a20_val,  method='auto')[1])
print(" 40 : ", stats.mannwhitneyu(AI_opto_d40_val,  AI_opto_a40_val,  method='auto')[1])
print(" 60 : ", stats.mannwhitneyu(AI_opto_d60_val,  AI_opto_a60_val,  method='auto')[1])
print(" 80 : ", stats.mannwhitneyu(AI_opto_d80_val,  AI_opto_a80_val,  method='auto')[1])
print("100 : ", stats.mannwhitneyu(AI_opto_d100_val, AI_opto_a100_val, method='auto')[1])


In [ ]:
# plot


fig, ((ax1)) = plt.subplots(1, 1, sharex=False, sharey=True, figsize=(5,4))
xpos1 = np.array([15,20,40,60,80,100])
xpos2 = np.array([15,20,40,60,80,100])

## Adults
yvals = [AI_opto_a15_mean, AI_opto_a20_mean, AI_opto_a40_mean, AI_opto_a60_mean, AI_opto_a80_mean, AI_opto_a100_mean]
yerrs = [AI_opto_a15_sem, AI_opto_a20_sem, AI_opto_a40_sem, AI_opto_a60_sem, AI_opto_a80_sem, AI_opto_a100_sem]
ax1.errorbar(xpos1, yvals, yerr=yerrs, c="#1F77B4", lw=2.5, ls='solid', capsize=2.5, elinewidth=2.5, label="Wild-type adults")

## Dauers
yvals = [AI_opto_d15_mean, AI_opto_d20_mean, AI_opto_d40_mean, AI_opto_d60_mean, AI_opto_d80_mean, AI_opto_d100_mean]
yerrs = [AI_opto_d15_sem, AI_opto_d20_sem, AI_opto_d40_sem, AI_opto_d60_sem, AI_opto_d80_sem, AI_opto_d100_sem]
ax1.errorbar(xpos2, yvals, yerr=yerrs, c="#FF7F0E", lw=2.5, ls='solid', capsize=2.5, elinewidth=2.5, label="Wild-type dauers") 

ax1.set_xlim(10,103)
ax1.set_ylim(0,1.05)
ax1.set_xticks(xpos2)
ax1.set_xticklabels(["15","20","40","60","80","100"], fontsize=12.5, rotation=45, fontname='sans-serif')
ax1.set_xlabel("Light intensity (A.U.)", fontsize=14, rotation=0, fontname='sans-serif')

ax1.set_yticks(np.arange(0,1.1,0.2))
ax1.set_yticklabels(["0.0","0.2","0.4","0.6","0.8","1.0"], fontsize=12.5, rotation=0, fontname='sans-serif')
ax1.set_ylabel("Avoidance index", fontsize=14, rotation=90, fontname='sans-serif')

ax1.text(15, 0.4, "**", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
ax1.text(20, 0.5, "*", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
ax1.text(60, 0.9, "*", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')


ax1.legend(bbox_to_anchor=(1.02,0.02), loc='lower right', fontsize=14)
ax1.spines[['right','top']].set_visible(False)
fig.tight_layout(pad=0.5)

plt.show()


## Behavior video

In [ ]:
import cv2

In [ ]:
# video to numpy array

cap = cv2.VideoCapture(r'your_directory\your_video.avi')  # change
# cap.isOpened()
test = cap.read()


FRAME = np.zeros((test[1].shape[0],test[1].shape[1],1))


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print('Done')
        break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    FRAME = np.copy(np.dstack((FRAME,gray)))

cap.release()


In [ ]:
FRAME = np.copy(FRAME[:,:,1:])  # remove first frame (0 array)
print(FRAME.shape)

In [ ]:
font1 = {'family': 'Arial',
         'color':  'red',
         'weight': 'bold',
         'size': 16,
         'alpha': 1.0}

In [ ]:
from PIL import Image, ImageEnhance

In [ ]:

for ii in range((FRAME.shape[2])):
    
    Fr = FRAME[90:-90,:,ii]
    
    if ii > 180 and ii <= 270:  # change
        fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
        ax1.imshow(Fr, cmap="gray")
#         ax1.text(380, 70, "LIGHT ON", fontdict=font1)
        ax1.axis("off")
        plt.show()
        
    else:
#         continue
        fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
        ax1.imshow(Fr, cmap="gray")
        ax1.axis("off")
        plt.show()
    
    print("Frame ",(ii))


In [ ]:
# save each frame as image

for ii in tqdm(range((FRAME.shape[2]))):
    
    Fr = FRAME[:,:,ii]
    
    if ii > 150 and ii <= 240:  # change
#         plt.figure(figsize=(10,10))
#         plt.text(370, 70, "LIGHT ON", fontdict=font1)
#         plt.axis("off")
#         plt.imsave(r'your_directory\fr%03d.png' % (ii),
#                    Fr, dpi=500, cmap="gray")  # change name / cmap (bone, seismic_r, gray, ...)
#         plt.close()
        
        fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
        ax1.imshow(Fr, cmap="gray")
        ax1.text(380, 70, "LIGHT ON", fontdict=font1)
        ax1.axis("off")
        plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
        plt.close()
        
    else:
#         plt.figure(figsize=(10,10))
#         plt.axis("off")
#         plt.imsave(r'your_directory\fr%03d.png' % (ii),
#                    Fr, dpi=500, cmap="gray")  # change name / cmap (bone, seismic_r, gray, ...)
#         plt.close()
        
        fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
        ax1.imshow(Fr, cmap="gray")
        ax1.axis("off")
        plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
        plt.close()


print("Done")


In [ ]:
# (temp) save frames as images (partial)

Fr_part = np.copy(FRAME[90:-90,:,68:129])  # change idx2 range


for ii in tqdm(range((Fr_part.shape[2]))):
    
    Fr = Fr_part[:,:,ii]
    
    fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
    ax1.imshow(Fr, cmap="gray")
    ax1.axis("off")
    plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight', pad_inches=0)  # change name
    plt.close()

print("Done")


In [ ]:
font2 = {'family': 'Arial',
         'color':  'red',
         'weight': 'bold',
         'size': 14,
         'alpha': 1.0}

In [ ]:
## =========================================================================================================================================================================
## save each frame as image (Reversal number mark ver.) ====================================================================================================================


## Adult

# for ii in tqdm(range((FRAME.shape[2]))):
    
#     Fr = FRAME[:,:,ii]
    
#     if ii > 157 and ii <= 217:  # change
#         fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
#         ax1.imshow(Fr, cmap="gray")
#         ax1.text(380, 70, "LIGHT ON", fontdict=font1)
#         ax1.axis("off")
#         plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
#         plt.close()
    
#     elif ii > 217 and ii <= 247:
#         fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
#         ax1.imshow(Fr, cmap="gray")
#         ax1.text(380, 70, "LIGHT ON", fontdict=font1)
#         ax1.text(185, 216, "1", fontdict=font2)
#         ax1.axis("off")
#         plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
#         plt.close()
    
#     elif ii > 247 and ii <= 257:
#         fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
#         ax1.imshow(Fr, cmap="gray")
#         ax1.text(185, 216, "1", fontdict=font2)
#         ax1.axis("off")
#         plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
#         plt.close()
    
#     elif ii > 257 and ii <= 307:
#         fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
#         ax1.imshow(Fr, cmap="gray")
#         ax1.text(185, 216, "1", fontdict=font2)
#         ax1.text(185, 290, "2", fontdict=font2)
#         ax1.axis("off")
#         plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
#         plt.close()
    
#     elif ii > 307:
#         fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
#         ax1.imshow(Fr, cmap="gray")
#         ax1.text(185, 216, "1", fontdict=font2)
#         ax1.text(185, 290, "2", fontdict=font2)
#         ax1.text(185, 364, "3", fontdict=font2)
#         ax1.axis("off")
#         plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
#         plt.close()
        
#     else:
#         fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
#         ax1.imshow(Fr, cmap="gray")
#         ax1.axis("off")
#         plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
#         plt.close()



## Dauer

for ii in tqdm(range((FRAME.shape[2]))):
    
    Fr = FRAME[:,:,ii]
    
    if ii > 150 and ii <= 211:  # change
        fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
        ax1.imshow(Fr, cmap="gray")
        ax1.text(380, 70, "LIGHT ON", fontdict=font1)
        ax1.axis("off")
        plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
        plt.close()
    
    elif ii > 211 and ii <= 240:
        fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
        ax1.imshow(Fr, cmap="gray")
        ax1.text(380, 70, "LIGHT ON", fontdict=font1)
        ax1.text(320, 320, "1", fontdict=font2)
        ax1.axis("off")
        plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
        plt.close()
    
    elif ii > 240:
        fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
        ax1.imshow(Fr, cmap="gray")
        ax1.text(320, 320, "1", fontdict=font2)
        ax1.axis("off")
        plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
        plt.close()
        
    else:
        fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,5))
        ax1.imshow(Fr, cmap="gray")
        ax1.axis("off")
        plt.savefig(r'your_directory\fr%03d.png' % (ii), facecolor='w', bbox_inches='tight')  # change name
        plt.close()



print("Done")

## =========================================================================================================================================================================
## =========================================================================================================================================================================

In [ ]:
# read frames

fr_dir = r'your_directory'  # change

fr_list = []
for file in os.listdir(fr_dir):
    if file.endswith(".png"):
        fr_list.append(os.path.join(fr_dir, file))
print(len(fr_list), "\n", fr_list[0])


In [ ]:
# plot (temp)

fr_img = cv2.imread(fr_list[220])  # change

fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(6,6))
ax1.imshow(fr_img)
ax1.set_xticks([])
ax1.set_yticks([])
# plt.axis("off")
plt.show()

print(fr_img.shape)


In [ ]:
# to video

w = int(fr_img.shape[1])
h = int(fr_img.shape[0])
fps = 30
delay = int(1000/fps)

# object for saving frames
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(r'your_directory\your_video.mp4',\
                      fourcc, fps, (w, h), True)  # 'False' for grayscale(2-D) video  //  change name

## Adult
# for ii in tqdm(range(len(fr_list))):
#     if ii > 66 and ii < 428:  # change
#         out.write(cv2.imread(fr_list[ii]))
#     else:
#         continue

## Dauer
for ii in tqdm(range(len(fr_list))):
    if ii > 59 and ii < 421:  # change
        out.write(cv2.imread(fr_list[ii]))
    else:
        continue

out.release()


### Image merge

#### Adult - Dauer - HD Cx path

In [ ]:
# Read image

import cv2


img_dir = r"your_directory\fr000.png"  # change filename

img_temp = cv2.imread(img_dir, cv2.IMREAD_UNCHANGED)


In [ ]:
# show image

plt.figure(figsize=(8,8))
plt.imshow(img_temp, cmap='gray')
plt.axis("off")

plt.show()

print(img_temp.shape)


In [ ]:
test_bi = np.copy(img_temp)

test_bi[test_bi > 80] = 255



plt.figure(figsize=(8,8))
plt.imshow(test_bi, cmap='gray')
# plt.imshow(test_bi[97:132,125:145,:], cmap='gray')

plt.text(109, 30, "X", ha='center', va='center', color='#FF7F0E', fontsize=18, fontname='sans-serif')
# plt.axis("off")

plt.show()


In [ ]:
# template array

template1 = np.zeros(test_bi.shape).astype(np.uint8)

template1[:,:] = 255  # white background


In [ ]:
# fill

# template1[100:180,130:220,:] = np.copy(test_bi[100:180,130:220,:])  # Adult 01
# template1[97:132,125:145,:] = np.copy(test_bi[97:132,125:145,:])  # Dauer 01
# template1[40:110,90:160,:] = np.copy(test_bi[40:110,90:160,:])  # HD Cx 02


In [ ]:
plt.figure()
plt.imshow(template1, cmap='gray')

plt.text(90, 200, "X", ha='center', va='center', color='#1F77B4', fontsize=18, fontname='sans-serif')  # Adult start
plt.text(132, 153, "X", ha='center', va='center', color='#FF7F0E', fontsize=18, fontname='sans-serif')  # Dauer start
plt.text(109, 30, "X", ha='center', va='center', color='#8C564B', fontsize=18, fontname='sans-serif')  # HD Cx start

plt.axis("off")

plt.show()

#### Adult - Dauer - BD Cx path

In [ ]:
# Read image

import cv2


img_dir = r"your_directory\fr022.png"  # change filename

img_temp = cv2.imread(img_dir, cv2.IMREAD_UNCHANGED)


In [ ]:
# show image

plt.figure(figsize=(8,8))
plt.imshow(img_temp, cmap='gray')
plt.axis("off")

plt.show()

print(img_temp.shape)


In [ ]:
test_bi = np.copy(img_temp)

test_bi[test_bi > 76] = 255



plt.figure(figsize=(8,8))
plt.imshow(test_bi, cmap='gray')
# plt.imshow(test_bi[47:85,213:245,:], cmap='gray')

# plt.text(230, 80, "X", ha='center', va='center', color='#FF7F0E', fontsize=18, fontname='sans-serif')

# plt.axis("off")

plt.show()


In [ ]:
# template array

template2 = np.zeros(test_bi.shape).astype(np.uint8)

template2[:,:] = 255  # white background


In [ ]:
# fill

# template2[118:155,212:245,:] = np.copy(test_bi[48:85,212:245,:])  # Adult 04
# template2[110:136,105:133,:] = np.copy(test_bi[110:136,105:133,:])  # Dauer 05
# template2[53:100,158:194,:] = np.copy(test_bi[53:100,158:194,:])  # BD Cx 03


In [ ]:
plt.figure()
plt.imshow(template2, cmap='gray')

plt.text(128, 190, "X", ha='center', va='center', color='#1F77B4', fontsize=18, fontname='sans-serif')  # Adult start
plt.text(141, 95, "X", ha='center', va='center', color='#FF7F0E', fontsize=18, fontname='sans-serif')  # Dauer start
plt.text(230, 80, "X", ha='center', va='center', color='#8C564B', fontsize=18, fontname='sans-serif')  # BD Cx start

plt.axis("off")

plt.show()

---